# **Step 8 : Recommandations System**

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Any
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import backend as K
from collections import Counter
import keras_tuner as kt
from tensorflow.keras import backend as K
from tensorflow.keras.saving import register_keras_serializable
import os
import json
import pickle
import joblib
from tensorflow.keras.models import load_model

In [2]:
class ImprovedSlidingWindowModel:
    def __init__(self, df, sequence_length=10, forecast_horizon=3):
        
        self.df = df.copy()
        self.forecast_horizon = forecast_horizon
        self.sequence_length = sequence_length
        self.scaler_X = None
        self.label_encoder = None
        self.model = None
        self.class_weights = None
        self.tuner = None
        self.best_hps = None
        
        # Improved thresholds for better balance
        self.minority_threshold = 0.08   # Very rare classes
        self.balanced_threshold = 0.20   # Medium frequency classes
        # Above 20% = majority classes
        
    def preprocess_data(self, target_col='weather_condition'):
        
        print("🧹 Enhanced preprocessing...")
        df = self.df.copy()
        
        # Handle missing values
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if col != target_col:
                df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
                df[col] = df[col].interpolate(method='linear')
                df[col].fillna(df[col].median(), inplace=True)
        
        # Remove rows with missing target
        df = df.dropna(subset=[target_col])
        # 🔄 Merge specific weather classes
        df[target_col] = df[target_col].replace({
            'Mainly Clear 🌤': 'Partly Clear 🌤/⛅',
            'Partly Cloudy ⛅': 'Partly Clear 🌤/⛅'
        })
        
        # Three-tier class analysis
        class_counts = df[target_col].value_counts()
        total_samples = len(df)
        print(f"Original class distribution:\n{class_counts}")
        print(f"Class percentages:\n{(class_counts/total_samples*100).round(2)}")
        
        # Categorize classes into three tiers
        self.minority_classes = []    # < 8%
        self.balanced_classes = []    # 8% - 20%  
        self.majority_classes = []    # > 20%
        
        for class_name, count in class_counts.items():
            percentage = count / total_samples
            if percentage < self.minority_threshold:
                self.minority_classes.append(class_name)
            elif percentage < self.balanced_threshold:
                self.balanced_classes.append(class_name)
            else:
                self.majority_classes.append(class_name)
        
        print(f"\nMinority classes (<{self.minority_threshold*100}%): {self.minority_classes}")
        print(f"Balanced classes ({self.minority_threshold*100}%-{self.balanced_threshold*100}%): {self.balanced_classes}")
        print(f"Majority classes (>{self.balanced_threshold*100}%): {self.majority_classes}")
        
        # Encode target
        self.label_encoder = LabelEncoder()
        df[target_col + '_encoded'] = self.label_encoder.fit_transform(df[target_col])
        
        # Feature engineering
        feature_cols = [col for col in df.columns if col not in ['date', target_col, target_col + '_encoded']]
        
        # Scale features
        self.scaler_X = StandardScaler()
        df[feature_cols] = self.scaler_X.fit_transform(df[feature_cols])
        
        # Calculate class weights
        unique_classes = np.unique(df[target_col + '_encoded'])
        self.class_weights = compute_class_weight('balanced', classes=unique_classes, y=df[target_col + '_encoded'])
        self.class_weight_dict = dict(zip(unique_classes, self.class_weights))
        
        # Store processed data
        self.df = df
        self.feature_cols = feature_cols
        self.target_col = target_col + '_encoded'
        self.original_target_col = target_col
        
        return df

    def create_improved_sliding_windows(self):
        
        print(f"\n🪟 Creating improved sliding windows with controlled rebalancing...")
        
        X, y = [], []
        data = self.df
        features = data[self.feature_cols].values
        target = data[self.target_col].values
        
        # Get class IDs for each tier
        minority_class_ids = [self.label_encoder.transform([cls])[0] 
                             for cls in self.minority_classes 
                             if cls in self.label_encoder.classes_]
        balanced_class_ids = [self.label_encoder.transform([cls])[0] 
                             for cls in self.balanced_classes 
                             if cls in self.label_encoder.classes_]
        majority_class_ids = [self.label_encoder.transform([cls])[0] 
                             for cls in self.majority_classes 
                             if cls in self.label_encoder.classes_]
        
        print(f"Minority class IDs: {minority_class_ids}")
        print(f"Balanced class IDs: {balanced_class_ids}")
        print(f"Majority class IDs: {majority_class_ids}")
        
        # Strategy 1: Intensive sampling for minority classes
        minority_windows = 0
        print("Creating intensive windows for minority classes...")
        
        for i in range(len(target)):
            if target[i] in minority_class_ids:
                # Create 7 overlapping windows around each minority event
                for offset in range(-3, 4):
                    start_idx = max(0, i - self.sequence_length + offset)
                    end_idx = start_idx + self.sequence_length + self.forecast_horizon
                    
                    if end_idx <= len(data):
                        X_seq = features[start_idx:start_idx + self.sequence_length]
                        y_seq = target[start_idx + self.sequence_length:end_idx]
                        
                        if len(X_seq) == self.sequence_length and len(y_seq) == self.forecast_horizon:
                            X.append(X_seq)
                            y.append(y_seq)
                            minority_windows += 1
        
        # Strategy 2: Moderate sampling for balanced classes
        balanced_windows = 0
        print("Creating moderate windows for balanced classes...")
        
        for i in range(0, len(data) - self.sequence_length - self.forecast_horizon + 1, 2):
            y_seq = target[i + self.sequence_length:i + self.sequence_length + self.forecast_horizon]
            
            # Check if sequence contains balanced classes
            if any(cls_id in balanced_class_ids for cls_id in y_seq):
                X_seq = features[i:i + self.sequence_length]
                X.append(X_seq)
                y.append(y_seq)
                balanced_windows += 1
        
        # Strategy 3: Controlled sampling for majority classes
        majority_windows = 0
        print("Creating controlled windows for majority classes...")
        
        # Calculate target number of majority windows (don't let them dominate)
        target_majority_windows = max(minority_windows * 1.5, balanced_windows * 0.8)
        
        majority_stride = max(3, int((len(data) - self.sequence_length - self.forecast_horizon) / target_majority_windows))
        
        for i in range(0, len(data) - self.sequence_length - self.forecast_horizon + 1, majority_stride):
            if majority_windows >= target_majority_windows:
                break
                
            y_seq = target[i + self.sequence_length:i + self.sequence_length + self.forecast_horizon]
            
            # Only add if it's primarily majority class and we haven't hit our limit
            if all(cls_id in majority_class_ids for cls_id in y_seq):
                X_seq = features[i:i + self.sequence_length]
                X.append(X_seq)
                y.append(y_seq)
                majority_windows += 1
        
        # Strategy 4: Add transitional windows (sequences that show class changes)
        transition_windows = 0
        print("Adding transitional windows...")
        
        for i in range(len(target) - self.forecast_horizon):
            current_class = target[i]
            future_classes = target[i+1:i+self.forecast_horizon+1]
            
            # Look for transitions from/to minority classes
            if (current_class in minority_class_ids or any(cls in minority_class_ids for cls in future_classes)):
                start_idx = max(0, i - self.sequence_length + 1)
                if start_idx + self.sequence_length + self.forecast_horizon <= len(data):
                    X_seq = features[start_idx:start_idx + self.sequence_length]
                    y_seq = target[start_idx + self.sequence_length:start_idx + self.sequence_length + self.forecast_horizon]
                    
                    if len(X_seq) == self.sequence_length and len(y_seq) == self.forecast_horizon:
                        X.append(X_seq)
                        y.append(y_seq)
                        transition_windows += 1
        
        X = np.array(X)
        y = np.array(y)
        
        print(f"\nWindow creation results:")
        print(f"Minority class windows: {minority_windows}")
        print(f"Balanced class windows: {balanced_windows}")
        print(f"Majority class windows: {majority_windows}")
        print(f"Transition windows: {transition_windows}")
        print(f"Total windows: {len(X)}")
        
        # Analyze final distribution
        final_dist = Counter(y.flatten())
        total_predictions = len(y.flatten())
        print(f"\nFinal class distribution:")
        for class_id, count in sorted(final_dist.items()):
            class_name = self.label_encoder.inverse_transform([class_id])[0]
            percentage = (count / total_predictions) * 100
            print(f"  {class_name}: {count} ({percentage:.1f}%)")
        
        return X, y
    
    def add_targeted_augmentation(self, X, y, augment_factor=1):
        
        print(f"\n🔄 Adding targeted augmentation for minority classes...")
        
        minority_class_ids = [self.label_encoder.transform([cls])[0] 
                             for cls in self.minority_classes 
                             if cls in self.label_encoder.classes_]
        
        X_aug = list(X)
        y_aug = list(y)
        augmented_count = 0
        
        for i in range(len(X)):
            # Check if sequence contains minority classes
            if any(cls_id in minority_class_ids for cls_id in y[i]):
                for _ in range(augment_factor):
                    # Add controlled noise to features
                    noise_scale = 0.01  # Very small noise
                    noise = np.random.normal(0, noise_scale, X[i].shape)
                    X_noisy = X[i] + noise
                    
                    # Add small temporal shifts occasionally
                    if np.random.random() < 0.3:
                        shift = np.random.randint(-1, 2)
                        if shift != 0:
                            X_shifted = np.roll(X[i], shift, axis=0)
                            X_aug.append(X_shifted)
                        else:
                            X_aug.append(X_noisy)
                    else:
                        X_aug.append(X_noisy)
                    
                    y_aug.append(y[i])
                    augmented_count += 1
        
        print(f"Added {augmented_count} augmented sequences for minority classes")
        return np.array(X_aug), np.array(y_aug)
    
    def focal_loss(self, alpha=0.25, gamma=2.0):
        
        def focal_loss_fn(y_true, y_pred):
            epsilon = K.epsilon()
            y_pred = K.clip(y_pred, epsilon, 1. - epsilon)
            
            # Cross entropy
            ce = -y_true * K.log(y_pred)
            
            # Focal weight: (1-p)^gamma
            p_t = K.sum(y_true * y_pred, axis=-1, keepdims=True)
            focal_weight = K.pow((1 - p_t), gamma)
            
            # Alpha weighting
            alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
            
            # Final focal loss
            focal_loss = alpha_t * focal_weight * ce
            
            return K.mean(K.sum(focal_loss, axis=-1))
        
        return focal_loss_fn
    
    def build_enhanced_model(self, input_shape, num_classes):
        
        model = Sequential([
            # First LSTM layer - larger for pattern recognition
            LSTM(256, return_sequences=True, input_shape=input_shape,
                 kernel_regularizer=l2(0.001), 
                 dropout=0.2, recurrent_dropout=0.2),
            BatchNormalization(),
            
            # Second LSTM layer - medium size
            LSTM(128, return_sequences=True,
                 kernel_regularizer=l2(0.001),
                 dropout=0.2, recurrent_dropout=0.2),
            BatchNormalization(),
            
            # Third LSTM layer - smaller for final processing
            LSTM(64, return_sequences=False,
                 kernel_regularizer=l2(0.001),
                 dropout=0.2, recurrent_dropout=0.2),
            BatchNormalization(),
            Dropout(0.3),
            
            # Dense layers with progressive size reduction
            Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
            BatchNormalization(),
            Dropout(0.4),
            
            Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
            Dropout(0.3),
            
            Dense(64, activation='relu'),
            Dropout(0.2),
            
            # Output layer
            Dense(self.forecast_horizon * num_classes, activation='softmax'),
        ])
        
        # Reshape to (batch_size, forecast_horizon, num_classes)
        model.add(tf.keras.layers.Reshape((self.forecast_horizon, num_classes)))
        return model
    
    def build_hypermodel(self, hp):
        
        # Tunable hyperparameters
        lstm1_units = hp.Int('lstm1_units', min_value=64, max_value=512, step=64)
        lstm2_units = hp.Int('lstm2_units', min_value=32, max_value=256, step=32)
        lstm3_units = hp.Int('lstm3_units', min_value=16, max_value=128, step=16)
        
        dense1_units = hp.Int('dense1_units', min_value=64, max_value=512, step=64)
        dense2_units = hp.Int('dense2_units', min_value=32, max_value=256, step=32)
        dense3_units = hp.Int('dense3_units', min_value=16, max_value=128, step=16)
        
        dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1)
        recurrent_dropout = hp.Float('recurrent_dropout', min_value=0.1, max_value=0.4, step=0.1)
        l2_reg = hp.Float('l2_reg', min_value=1e-4, max_value=1e-2, sampling='LOG')
        
        learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')
        focal_alpha = hp.Float('focal_alpha', min_value=0.1, max_value=0.5, step=0.1)
        focal_gamma = hp.Float('focal_gamma', min_value=1.0, max_value=3.0, step=0.5)
        
        num_classes = len(self.label_encoder.classes_)
        input_shape = (self.sequence_length, len(self.feature_cols))
        
        model = Sequential([
            # First LSTM layer
            LSTM(lstm1_units, return_sequences=True, input_shape=input_shape,
                 kernel_regularizer=l2(l2_reg), 
                 dropout=dropout_rate, recurrent_dropout=recurrent_dropout),
            BatchNormalization(),
            
            # Second LSTM layer
            LSTM(lstm2_units, return_sequences=True,
                 kernel_regularizer=l2(l2_reg),
                 dropout=dropout_rate, recurrent_dropout=recurrent_dropout),
            BatchNormalization(),
            
            # Third LSTM layer
            LSTM(lstm3_units, return_sequences=False,
                 kernel_regularizer=l2(l2_reg),
                 dropout=dropout_rate, recurrent_dropout=recurrent_dropout),
            BatchNormalization(),
            Dropout(dropout_rate),
            
            # Dense layers
            Dense(dense1_units, activation='relu', kernel_regularizer=l2(l2_reg)),
            BatchNormalization(),
            Dropout(dropout_rate + 0.1),
            
            Dense(dense2_units, activation='relu', kernel_regularizer=l2(l2_reg)),
            Dropout(dropout_rate),
            
            Dense(dense3_units, activation='relu'),
            Dropout(dropout_rate - 0.1),
            
            # Output layer
            Dense(self.forecast_horizon * num_classes, activation='softmax'),
        ])
        
        # Reshape to (batch_size, forecast_horizon, num_classes)
        model.add(tf.keras.layers.Reshape((self.forecast_horizon, num_classes)))
        
        # Compile with tuned hyperparameters
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
            loss=self.focal_loss(alpha=focal_alpha, gamma=focal_gamma),
            metrics=['accuracy']
        )
        
        return model
    
    def hyperparameter_tuning(self, X_train, y_train, max_trials=20):
        
        print("\n" + "="*70)
        print("🔧 HYPERPARAMETER TUNING")
        print("="*70)
        
        # Initialize tuner
        self.tuner = kt.RandomSearch(
            self.build_hypermodel,
            objective='val_accuracy',
            max_trials=max_trials,
            directory='weather_tuning',
            project_name='lstm_weather_forecast'
        )
        
        # Search callbacks
        stop_early = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
        print(f"Starting hyperparameter search with {max_trials} trials...")
        print("This may take a while depending on your hardware...")
        
        # Perform search
        self.tuner.search(
            X_train, y_train,
            epochs=15,
            batch_size=32,
            validation_split=0.2,
            callbacks=[stop_early],
            verbose=1
        )
        
        # Get best hyperparameters
        self.best_hps = self.tuner.get_best_hyperparameters(num_trials=1)[0]
        
        print("\n🏆 BEST HYPERPARAMETERS FOUND:")
        print("="*50)
        print(f"LSTM Layer 1 Units: {self.best_hps.get('lstm1_units')}")
        print(f"LSTM Layer 2 Units: {self.best_hps.get('lstm2_units')}")
        print(f"LSTM Layer 3 Units: {self.best_hps.get('lstm3_units')}")
        print(f"Dense Layer 1 Units: {self.best_hps.get('dense1_units')}")
        print(f"Dense Layer 2 Units: {self.best_hps.get('dense2_units')}")
        print(f"Dense Layer 3 Units: {self.best_hps.get('dense3_units')}")
        print(f"Dropout Rate: {self.best_hps.get('dropout_rate'):.3f}")
        print(f"Recurrent Dropout: {self.best_hps.get('recurrent_dropout'):.3f}")
        print(f"L2 Regularization: {self.best_hps.get('l2_reg'):.6f}")
        print(f"Learning Rate: {self.best_hps.get('learning_rate'):.6f}")
        print(f"Focal Loss Alpha: {self.best_hps.get('focal_alpha'):.3f}")
        print(f"Focal Loss Gamma: {self.best_hps.get('focal_gamma'):.1f}")
        
        # Build best model
        self.model = self.tuner.hypermodel.build(self.best_hps)
        
        return self.best_hps
    
    def train_improved_model(self, use_augmentation=True, use_hypertuning=True, max_trials=20):
        
        print("\n" + "="*70)
        print("TRAINING IMPROVED SLIDING WINDOW + REBALANCING MODEL")
        if use_hypertuning:
            print("WITH HYPERPARAMETER TUNING")
        print("="*70)
        
        # Create improved sliding windows
        X, y = self.create_improved_sliding_windows()
        
        # Add targeted augmentation if requested
        if use_augmentation:
            X, y = self.add_targeted_augmentation(X, y, augment_factor=1)
        
        num_classes = len(self.label_encoder.classes_)
        
        if len(X) == 0:
            raise ValueError("No sequences created!")
        
        print(f"\nFinal training data: X={X.shape}, y={y.shape}")
        print(f"Number of classes: {num_classes}")
        
        # Convert to categorical
        y_onehot = to_categorical(y, num_classes=num_classes)
        
        # Stratified train-test split
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, test_idx = next(sss.split(X, y[:, 0]))
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_onehot[train_idx], y_onehot[test_idx]
        
        print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
        
        # Hyperparameter tuning
        if use_hypertuning:
            self.hyperparameter_tuning(X_train, y_train, max_trials=max_trials)
        else:
            # Build default model
            self.model = self.build_enhanced_model((self.sequence_length, X.shape[2]), num_classes)
            
            # Use default focal loss
            self.model.compile(
                optimizer=Adam(learning_rate=0.0005, clipnorm=1.0),
                loss=self.focal_loss(alpha=0.25, gamma=2.0),
                metrics=['accuracy']
            )
        
        # Enhanced callbacks
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=10, min_lr=1e-7, verbose=1)
        ]
        
        # Train model with best hyperparameters
        print(f"\nTraining {'hypertuned' if use_hypertuning else 'default'} model...")
        history = self.model.fit(
            X_train, y_train,
            epochs=200,
            batch_size=32,
            validation_split=0.2,
            callbacks=callbacks,
            verbose=1
        )
        
        # Evaluate
        print(f"\nEvaluating {'hypertuned' if use_hypertuning else 'default'} model...")
        y_pred_prob = self.model.predict(X_test, verbose=0)
        y_pred = np.argmax(y_pred_prob, axis=-1)
        y_true = np.argmax(y_test, axis=-1)
        
        return self.evaluate_improved_results(y_true, y_pred, history, use_hypertuning)
    
    def evaluate_improved_results(self, y_true, y_pred, history, hypertuned=False):
        """
        Comprehensive evaluation with class-tier analysis
        """
        model_type = "HYPERTUNED" if hypertuned else "DEFAULT"
        print("\n" + "="*70)
        print(f"IMPROVED SLIDING WINDOW + REBALANCING RESULTS ({model_type})")
        print("="*70)
        
        # Overall accuracy
        overall_acc = accuracy_score(y_true.flatten(), y_pred.flatten())
        print(f"Overall Test Accuracy: {overall_acc:.4f}")
        
        # Daily accuracies
        daily_accuracies = []
        for day in range(self.forecast_horizon):
            day_acc = accuracy_score(y_true[:, day], y_pred[:, day])
            daily_accuracies.append(day_acc)
            print(f"Day {day+1} accuracy: {day_acc:.4f}")
        
        # Detailed classification report for Day 1
        print(f"\nDetailed Classification Report (Day 1):")
        print("-" * 70)
        target_names = self.label_encoder.classes_
        report = classification_report(y_true[:, 0], y_pred[:, 0], 
                                     target_names=target_names, 
                                     zero_division=0,
                                     output_dict=True)
        print(classification_report(y_true[:, 0], y_pred[:, 0], 
                                  target_names=target_names, 
                                  zero_division=0))
        
        # Class-tier performance analysis
        print(f"\n🎯 CLASS-TIER PERFORMANCE ANALYSIS:")
        print("="*70)
        
        for tier_name, class_list in [
            ("MINORITY", self.minority_classes),
            ("BALANCED", self.balanced_classes), 
            ("MAJORITY", self.majority_classes)
        ]:
            print(f"\n{tier_name} CLASSES:")
            print("-" * 40)
            tier_f1_scores = []
            tier_recalls = []
            tier_precisions = []
            
            for class_name in class_list:
                if class_name in report:
                    metrics = report[class_name]
                    f1 = metrics['f1-score']
                    precision = metrics['precision']
                    recall = metrics['recall']
                    support = metrics['support']
                    
                    tier_f1_scores.append(f1)
                    tier_recalls.append(recall)
                    tier_precisions.append(precision)
                    
                    status = "✅" if f1 > 0.3 else "⚠️" if f1 > 0.1 else "❌"
                    print(f"  {status} {class_name:20s}: P={precision:.3f}, R={recall:.3f}, F1={f1:.3f}, N={support}")
            
            if tier_f1_scores:
                avg_f1 = np.mean(tier_f1_scores)
                avg_precision = np.mean(tier_precisions)
                avg_recall = np.mean(tier_recalls)
                print(f"\n  📊 {tier_name} AVERAGES:")
                print(f"     Precision: {avg_precision:.4f}")
                print(f"     Recall: {avg_recall:.4f}")
                print(f"     F1-Score: {avg_f1:.4f}")
        
        # Problem class identification
        print(f"\n⚠️  CLASSES STILL STRUGGLING (F1 < 0.2):")
        print("-" * 50)
        struggling_classes = []
        for class_name in target_names:
            if class_name in report and report[class_name]['f1-score'] < 0.2:
                struggling_classes.append(class_name)
                f1 = report[class_name]['f1-score']
                support = report[class_name]['support']
                print(f"  ❌ {class_name}: F1={f1:.3f}, N={support}")
        
        if not struggling_classes:
            print("  🎉 No classes with F1 < 0.2!")
        
        # Success stories
        print(f"\n🎉 BIGGEST IMPROVEMENTS (F1 > 0.5):")
        print("-" * 50)
        success_classes = []
        for class_name in target_names:
            if class_name in report and report[class_name]['f1-score'] > 0.5:
                success_classes.append(class_name)
                f1 = report[class_name]['f1-score']
                support = report[class_name]['support']
                print(f"  ✅ {class_name}: F1={f1:.3f}, N={support}")
        
        # Hyperparameter summary if tuned
        if hypertuned and self.best_hps:
            print(f"\n🔧 HYPERPARAMETER SUMMARY:")
            print("-" * 50)
            print(f"  Best Learning Rate: {self.best_hps.get('learning_rate'):.6f}")
            print(f"  Best Dropout Rate: {self.best_hps.get('dropout_rate'):.3f}")
            print(f"  Best LSTM Units: {self.best_hps.get('lstm1_units')}-{self.best_hps.get('lstm2_units')}-{self.best_hps.get('lstm3_units')}")
            print(f"  Best Dense Units: {self.best_hps.get('dense1_units')}-{self.best_hps.get('dense2_units')}-{self.best_hps.get('dense3_units')}")
            print(f"  Best Focal Loss: α={self.best_hps.get('focal_alpha'):.3f}, γ={self.best_hps.get('focal_gamma'):.1f}")
        
        return overall_acc, daily_accuracies, history, y_true, y_pred
    
    def predict_future(self, recent_data):
        
        if isinstance(recent_data, pd.DataFrame):
            recent_scaled = self.scaler_X.transform(recent_data[self.feature_cols])
        elif isinstance(recent_data, np.ndarray):
            if recent_data.shape[1] == len(self.feature_cols):
                recent_scaled = self.scaler_X.transform(recent_data)
            else:
                recent_scaled = recent_data
        else:
            raise ValueError("recent_data must be either a DataFrame or numpy array")
        
        input_seq = recent_scaled[-self.sequence_length:].reshape(1, self.sequence_length, -1)
        pred_prob = self.model.predict(input_seq, verbose=0)
        pred_classes = np.argmax(pred_prob[0], axis=-1)
        
        predicted_conditions = self.label_encoder.inverse_transform(pred_classes)
        confidence_scores = np.max(pred_prob[0], axis=-1)
        
        # Calculate prediction uncertainty
        entropy = -np.sum(pred_prob[0] * np.log(pred_prob[0] + 1e-8), axis=-1)
        normalized_entropy = entropy / np.log(len(self.label_encoder.classes_))
        
        return predicted_conditions, confidence_scores, normalized_entropy

In [3]:
class OptimizedHumidityMinLSTM:
    def __init__(self, df, sequence_length=10, forecast_horizon=3):
        self.df = df.copy()
        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.scaler_features = None
        self.scaler_targets = None
        self.model = None
        self.best_params = None
        self.study = None
        
        # Temperature target columns
        self.target_cols = [
            'relative_humidity_2m_min (%)'
        ]
        
    def preprocess_data(self):
        print("🧹 Preprocessing data for max temperature prediction...")
        df = self.df.copy()
        
        # Check which target columns are available
        available_targets = [col for col in self.target_cols if col in df.columns]
        if not available_targets:
            raise ValueError(f"None of the target columns {self.target_cols} found in the dataframe")
        
        self.available_targets = available_targets
        print(f"Available target columns: {available_targets}")
        
        # Handle missing values for targets
        for col in available_targets:
            df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            df[col] = df[col].interpolate(method='linear')
            df[col].fillna(df[col].median(), inplace=True)
        
        # Remove rows with any missing target values
        df = df.dropna(subset=available_targets)
        
        # Define feature columns (exclude date, weather_condition, and target columns)
        exclude_cols = ['date', 'weather_condition', 'temperature_2m_max (°C)', 'relative_humidity_2m_max (%)', 'temperature_2m_min (°C)'] + available_targets
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        
        # Handle missing values for features
        numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            df[col] = df[col].interpolate(method='linear')
            df[col].fillna(df[col].median(), inplace=True)
        
        # Scale features
        self.scaler_features = StandardScaler()
        df[feature_cols] = self.scaler_features.fit_transform(df[feature_cols])
        
        # Scale targets
        self.scaler_targets = StandardScaler()
        df[available_targets] = self.scaler_targets.fit_transform(df[available_targets])
        
        # Store processed data
        self.df_processed = df
        self.feature_cols = feature_cols
        
        print(f"Feature columns ({len(feature_cols)}): {feature_cols[:5]}..." if len(feature_cols) > 5 else f"Feature columns: {feature_cols}")
        print(f"Target columns: {available_targets}")
        print(f"Data shape after preprocessing: {df.shape}")
        
        return df
    
    def create_sequences(self):
        print(f"🪟 Creating sequences for Maximum temperature prediction...")
        
        X, y = [], []
        data = self.df_processed
        features = data[self.feature_cols].values
        targets = data[self.available_targets].values
        
        # Create sequences with sliding window
        for i in range(len(data) - self.sequence_length - self.forecast_horizon + 1):
            # Input sequence
            X_seq = features[i:i + self.sequence_length]
            
            # Target sequence (next forecast_horizon days)
            y_seq = targets[i + self.sequence_length:i + self.sequence_length + self.forecast_horizon]
            
            X.append(X_seq)
            y.append(y_seq)
        
        X = np.array(X)
        y = np.array(y)
        
        print(f"Created sequences: X={X.shape}, y={y.shape}")
        print(f"Input sequence length: {self.sequence_length}")
        print(f"Forecast horizon: {self.forecast_horizon}")
        print(f"Number of features: {X.shape[2]}")
        print(f"Number of target variables: {y.shape[2]}")
        
        return X, y
    
    def build_model_with_params(self, trial, input_shape, output_shape):
        
        # Hyperparameters to optimize
        lstm1_units = trial.suggest_int('lstm1_units', 32, 256, step=32)
        lstm2_units = trial.suggest_int('lstm2_units', 16, 128, step=16)
        lstm3_units = trial.suggest_int('lstm3_units', 8, 64, step=8)
        
        dense1_units = trial.suggest_int('dense1_units', 32, 128, step=16)
        dense2_units = trial.suggest_int('dense2_units', 16, 64, step=8)
        
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
        recurrent_dropout = trial.suggest_float('recurrent_dropout', 0.1, 0.4, step=0.1)
        l2_reg = trial.suggest_float('l2_reg', 1e-5, 1e-2, log=True)
        
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        batch_norm = trial.suggest_categorical('batch_norm', [True, False])
        
        model = Sequential()
        
        # First LSTM layer
        model.add(LSTM(lstm1_units, return_sequences=True, input_shape=input_shape,
                      kernel_regularizer=l2(l2_reg), 
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        # Second LSTM layer
        model.add(LSTM(lstm2_units, return_sequences=True,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        # Third LSTM layer
        model.add(LSTM(lstm3_units, return_sequences=False,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(Dropout(dropout_rate + 0.1))
        
        # Dense layers
        model.add(Dense(dense1_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        if batch_norm:
            model.add(BatchNormalization())
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense2_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        model.add(Dropout(dropout_rate))
        
        # Output layer
        model.add(Dense(output_shape[0] * output_shape[1], activation='linear'))
        
        # Reshape to (forecast_horizon, num_target_variables)
        model.add(tf.keras.layers.Reshape(output_shape))
        
        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
            loss='mse',
            metrics=['mae']
        )
        
        return model
    
    def objective(self, trial):
        
        try:
            # Create sequences
            X, y = self.create_sequences()
            
            if len(X) == 0:
                return float('inf')
            
            # Train-validation split
            X_train, X_val, y_train, y_val = train_test_split(
                X, y, test_size=0.2, random_state=42, shuffle=True
            )
            
            # Build model with trial parameters
            input_shape = (self.sequence_length, X.shape[2])
            output_shape = (self.forecast_horizon, len(self.available_targets))
            
            model = self.build_model_with_params(trial, input_shape, output_shape)
            
            # Training parameters
            batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
            
            # Callbacks
            callbacks = [
                EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=0)
            ]
            
            # Train model
            history = model.fit(
                X_train, y_train,
                epochs=50,  # Reduced for faster optimization
                batch_size=batch_size,
                validation_data=(X_val, y_val),
                callbacks=callbacks,
                verbose=0
            )
            
            # Get best validation loss
            best_val_loss = min(history.history['val_loss'])
            
            # Clean up to free memory
            del model
            tf.keras.backend.clear_session()
            
            return best_val_loss
            
        except Exception as e:
            print(f"Trial failed with error: {e}")
            return float('inf')
    
    def optimize_hyperparameters(self, n_trials=20, timeout=None):
        
        print("\n" + "="*70)
        print("🔧 HYPERPARAMETER OPTIMIZATION WITH OPTUNA")
        print("="*70)
        
        # Create study
        self.study = optuna.create_study(direction='minimize', 
                                        study_name='temperature_lstm_optimization')
        
        print(f"Starting optimization with {n_trials} trials...")
        if timeout:
            print(f"Timeout set to {timeout} seconds")
        
        # Optimize
        self.study.optimize(self.objective, n_trials=n_trials, timeout=timeout)
        
        # Get best parameters
        self.best_params = self.study.best_params
        
        print("\n🏆 OPTIMIZATION COMPLETED!")
        print("="*50)
        print(f"Best validation loss: {self.study.best_value:.6f}")
        print(f"Number of trials: {len(self.study.trials)}")
        
        print(f"\n🎯 BEST HYPERPARAMETERS:")
        print("-" * 40)
        for key, value in self.best_params.items():
            print(f"{key:20s}: {value}")
        
        return self.best_params
    
    def build_best_model(self, input_shape, output_shape):
        """Build model with best parameters found by Optuna"""
        if self.best_params is None:
            raise ValueError("No optimization performed yet. Run optimize_hyperparameters() first.")
        
        model = Sequential()
        
        # Use best parameters
        lstm1_units = self.best_params['lstm1_units']
        lstm2_units = self.best_params['lstm2_units']
        lstm3_units = self.best_params['lstm3_units']
        dense1_units = self.best_params['dense1_units']
        dense2_units = self.best_params['dense2_units']
        dropout_rate = self.best_params['dropout_rate']
        recurrent_dropout = self.best_params['recurrent_dropout']
        l2_reg = self.best_params['l2_reg']
        learning_rate = self.best_params['learning_rate']
        batch_norm = self.best_params['batch_norm']
        
        # Build architecture
        model.add(LSTM(lstm1_units, return_sequences=True, input_shape=input_shape,
                      kernel_regularizer=l2(l2_reg), 
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(LSTM(lstm2_units, return_sequences=True,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(LSTM(lstm3_units, return_sequences=False,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense1_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        if batch_norm:
            model.add(BatchNormalization())
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense2_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        model.add(Dropout(dropout_rate))
        
        model.add(Dense(output_shape[0] * output_shape[1], activation='linear'))
        model.add(tf.keras.layers.Reshape(output_shape))
        
        # Compile with best parameters
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
            loss='mse',
            metrics=['mae']
        )
        
        return model
    
    def train_optimized_model(self, validation_split=0.2, epochs=200):
        
        print("\n" + "="*70)
        print("TRAINING OPTIMIZED TEMPERATURE MAX LSTM MODEL")
        print("="*70)
        
        if self.best_params is None:
            raise ValueError("No optimization performed yet. Run optimize_hyperparameters() first.")
        
        # Create sequences
        X, y = self.create_sequences()
        
        if len(X) == 0:
            raise ValueError("No sequences created!")
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, shuffle=True
        )
        
        print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
        
        # Build optimized model
        input_shape = (self.sequence_length, X.shape[2])
        output_shape = (self.forecast_horizon, len(self.available_targets))
        
        self.model = self.build_best_model(input_shape, output_shape)
        
        print(f"\nOptimized model architecture:")
        self.model.summary()
        
        # Use best batch size
        batch_size = self.best_params.get('batch_size', 32)
        
        # Callbacks
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)
        ]
        
        # Train model
        print(f"\nTraining optimized model...")
        history = self.model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            callbacks=callbacks,
            verbose=1
        )
        
        # Evaluate
        print(f"\nEvaluating optimized model...")
        y_pred = self.model.predict(X_test, verbose=0)
        
        return self.evaluate_model(y_test, y_pred, history)
    
    def evaluate_model(self, y_true, y_pred, history):
        print("\n" + "="*70)
        print("OPTIMIZED TEMPERATURE MAX LSTM RESULTS")
        print("="*70)
        
        # Inverse transform predictions and true values to original scale
        y_true_original = np.zeros_like(y_true)
        y_pred_original = np.zeros_like(y_pred)
        
        for day in range(self.forecast_horizon):
            y_true_original[:, day, :] = self.scaler_targets.inverse_transform(y_true[:, day, :])
            y_pred_original[:, day, :] = self.scaler_targets.inverse_transform(y_pred[:, day, :])
        
        # Calculate metrics for each target variable and forecast day
        results = {}
        
        for i, target_name in enumerate(self.available_targets):
            print(f"\n🌡️ {target_name}:")
            print("-" * 50)
            
            target_results = {
                'mae': [],
                'rmse': [],
                'r2': [],
                'mape': []
            }
            
            for day in range(self.forecast_horizon):
                y_true_day = y_true_original[:, day, i]
                y_pred_day = y_pred_original[:, day, i]
                
                mae = mean_absolute_error(y_true_day, y_pred_day)
                rmse = np.sqrt(mean_squared_error(y_true_day, y_pred_day))
                r2 = r2_score(y_true_day, y_pred_day)
                
                # Calculate MAPE (Mean Absolute Percentage Error)
                mape = np.mean(np.abs((y_true_day - y_pred_day) / (y_true_day + 1e-8))) * 100
                
                target_results['mae'].append(mae)
                target_results['rmse'].append(rmse)
                target_results['r2'].append(r2)
                target_results['mape'].append(mape)
                
                print(f"  Day {day+1}: MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f}, MAPE={mape:.2f}%")
            
            # Calculate averages
            avg_mae = np.mean(target_results['mae'])
            avg_rmse = np.mean(target_results['rmse'])
            avg_r2 = np.mean(target_results['r2'])
            avg_mape = np.mean(target_results['mape'])
            
            target_results['avg_mae'] = avg_mae
            target_results['avg_rmse'] = avg_rmse
            target_results['avg_r2'] = avg_r2
            target_results['avg_mape'] = avg_mape
            
            print(f"  Average: MAE={avg_mae:.3f}, RMSE={avg_rmse:.3f}, R²={avg_r2:.3f}, MAPE={avg_mape:.2f}%")
            
            results[target_name] = target_results
        
        # Overall performance summary
        print(f"\n📊 OVERALL PERFORMANCE SUMMARY:")
        print("="*50)
        
        all_maes = [results[target]['avg_mae'] for target in self.available_targets]
        all_rmses = [results[target]['avg_rmse'] for target in self.available_targets]
        all_r2s = [results[target]['avg_r2'] for target in self.available_targets]
        all_mapes = [results[target]['avg_mape'] for target in self.available_targets]
        
        print(f"Overall Average MAE: {np.mean(all_maes):.3f}")
        print(f"Overall Average RMSE: {np.mean(all_rmses):.3f}")
        print(f"Overall Average R²: {np.mean(all_r2s):.3f}")
        print(f"Overall Average MAPE: {np.mean(all_mapes):.2f}%")
        
        # Optimization summary
        if self.best_params:
            print(f"\n🔧 OPTIMIZATION SUMMARY:")
            print("-" * 40)
            print(f"Best validation loss: {self.study.best_value:.6f}")
            print(f"Number of trials completed: {len(self.study.trials)}")
            print(f"Best LSTM architecture: {self.best_params['lstm1_units']}-{self.best_params['lstm2_units']}-{self.best_params['lstm3_units']}")
            print(f"Best Dense architecture: {self.best_params['dense1_units']}-{self.best_params['dense2_units']}")
            print(f"Best learning rate: {self.best_params['learning_rate']:.6f}")
            print(f"Best dropout rate: {self.best_params['dropout_rate']:.3f}")
        
        return results, history, y_true_original, y_pred_original
    
    def predict_future(self, recent_data):
        if isinstance(recent_data, pd.DataFrame):
            recent_scaled = self.scaler_features.transform(recent_data[self.feature_cols])
        elif isinstance(recent_data, np.ndarray):
            if recent_data.shape[1] == len(self.feature_cols):
                recent_scaled = self.scaler_features.transform(recent_data)
            else:
                recent_scaled = recent_data
        else:
            raise ValueError("recent_data must be a DataFrame or NumPy array")

        input_seq = recent_scaled[-self.sequence_length:].reshape(1, self.sequence_length, -1)
        prediction_scaled = self.model.predict(input_seq, verbose=0)

        # Inverse transform
        prediction_original = np.zeros_like(prediction_scaled)
        for day in range(self.forecast_horizon):
            prediction_original[0, day, :] = self.scaler_targets.inverse_transform(
                prediction_scaled[0, day, :].reshape(1, -1)
            )

        predicted_values = prediction_original[0, :, 0]  

        return predicted_values
    
    def plot_training_history(self, history):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss
        ax1.plot(history.history['loss'], label='Training Loss')
        ax1.plot(history.history['val_loss'], label='Validation Loss')
        ax1.set_title('Optimized Model Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        
        # MAE
        ax2.plot(history.history['mae'], label='Training MAE')
        ax2.plot(history.history['val_mae'], label='Validation MAE')
        ax2.set_title('Optimized Model MAE')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('MAE')
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
    
    def plot_optimization_history(self):
        
        if self.study is None:
            print("No optimization study available to plot.")
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Optimization history
        values = [trial.value for trial in self.study.trials if trial.value is not None]
        ax1.plot(values)
        ax1.set_title('Optimization History')
        ax1.set_xlabel('Trial')
        ax1.set_ylabel('Validation Loss')
        ax1.grid(True)
        
        # Parameter importance (if available)
        try:
            importance = optuna.importance.get_param_importances(self.study)
            params = list(importance.keys())
            importances = list(importance.values())
            
            ax2.barh(params, importances)
            ax2.set_title('Parameter Importance')
            ax2.set_xlabel('Importance')
        except:
            ax2.text(0.5, 0.5, 'Parameter importance\nnot available', 
                    ha='center', va='center', transform=ax2.transAxes)
        
        plt.tight_layout()
        plt.show()

In [4]:
class OptimizedHumidityMaxLSTM:
    def __init__(self, df, sequence_length=10, forecast_horizon=3):
        self.df = df.copy()
        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.scaler_features = None
        self.scaler_targets = None
        self.model = None
        self.best_params = None
        self.study = None
        
        # Temperature target columns
        self.target_cols = [
            'relative_humidity_2m_max (%)'
        ]
        
    def preprocess_data(self):
        print("🧹 Preprocessing data for max temperature prediction...")
        df = self.df.copy()
        
        # Check which target columns are available
        available_targets = [col for col in self.target_cols if col in df.columns]
        if not available_targets:
            raise ValueError(f"None of the target columns {self.target_cols} found in the dataframe")
        
        self.available_targets = available_targets
        print(f"Available target columns: {available_targets}")
        
        # Handle missing values for targets
        for col in available_targets:
            df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            df[col] = df[col].interpolate(method='linear')
            df[col].fillna(df[col].median(), inplace=True)
        
        # Remove rows with any missing target values
        df = df.dropna(subset=available_targets)
        
        # Define feature columns (exclude date, weather_condition, and target columns)
        exclude_cols = ['date', 'weather_condition', 'temperature_2m_max (°C)', 'relative_humidity_2m_min (%)', 'temperature_2m_min (°C)'] + available_targets
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        
        # Handle missing values for features
        numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            df[col] = df[col].interpolate(method='linear')
            df[col].fillna(df[col].median(), inplace=True)
        
        # Scale features
        self.scaler_features = StandardScaler()
        df[feature_cols] = self.scaler_features.fit_transform(df[feature_cols])
        
        # Scale targets
        self.scaler_targets = StandardScaler()
        df[available_targets] = self.scaler_targets.fit_transform(df[available_targets])
        
        # Store processed data
        self.df_processed = df
        self.feature_cols = feature_cols
        
        print(f"Feature columns ({len(feature_cols)}): {feature_cols[:5]}..." if len(feature_cols) > 5 else f"Feature columns: {feature_cols}")
        print(f"Target columns: {available_targets}")
        print(f"Data shape after preprocessing: {df.shape}")
        
        return df
    
    def create_sequences(self):
        print(f"🪟 Creating sequences for Maximum temperature prediction...")
        
        X, y = [], []
        data = self.df_processed
        features = data[self.feature_cols].values
        targets = data[self.available_targets].values
        
        # Create sequences with sliding window
        for i in range(len(data) - self.sequence_length - self.forecast_horizon + 1):
            # Input sequence
            X_seq = features[i:i + self.sequence_length]
            
            # Target sequence (next forecast_horizon days)
            y_seq = targets[i + self.sequence_length:i + self.sequence_length + self.forecast_horizon]
            
            X.append(X_seq)
            y.append(y_seq)
        
        X = np.array(X)
        y = np.array(y)
        
        print(f"Created sequences: X={X.shape}, y={y.shape}")
        print(f"Input sequence length: {self.sequence_length}")
        print(f"Forecast horizon: {self.forecast_horizon}")
        print(f"Number of features: {X.shape[2]}")
        print(f"Number of target variables: {y.shape[2]}")
        
        return X, y
    
    def build_model_with_params(self, trial, input_shape, output_shape):
        
        # Hyperparameters to optimize
        lstm1_units = trial.suggest_int('lstm1_units', 32, 256, step=32)
        lstm2_units = trial.suggest_int('lstm2_units', 16, 128, step=16)
        lstm3_units = trial.suggest_int('lstm3_units', 8, 64, step=8)
        
        dense1_units = trial.suggest_int('dense1_units', 32, 128, step=16)
        dense2_units = trial.suggest_int('dense2_units', 16, 64, step=8)
        
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
        recurrent_dropout = trial.suggest_float('recurrent_dropout', 0.1, 0.4, step=0.1)
        l2_reg = trial.suggest_float('l2_reg', 1e-5, 1e-2, log=True)
        
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        batch_norm = trial.suggest_categorical('batch_norm', [True, False])
        
        model = Sequential()
        
        # First LSTM layer
        model.add(LSTM(lstm1_units, return_sequences=True, input_shape=input_shape,
                      kernel_regularizer=l2(l2_reg), 
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        # Second LSTM layer
        model.add(LSTM(lstm2_units, return_sequences=True,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        # Third LSTM layer
        model.add(LSTM(lstm3_units, return_sequences=False,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(Dropout(dropout_rate + 0.1))
        
        # Dense layers
        model.add(Dense(dense1_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        if batch_norm:
            model.add(BatchNormalization())
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense2_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        model.add(Dropout(dropout_rate))
        
        # Output layer
        model.add(Dense(output_shape[0] * output_shape[1], activation='linear'))
        
        # Reshape to (forecast_horizon, num_target_variables)
        model.add(tf.keras.layers.Reshape(output_shape))
        
        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
            loss='mse',
            metrics=['mae']
        )
        
        return model
    
    def objective(self, trial):
        
        try:
            # Create sequences
            X, y = self.create_sequences()
            
            if len(X) == 0:
                return float('inf')
            
            # Train-validation split
            X_train, X_val, y_train, y_val = train_test_split(
                X, y, test_size=0.2, random_state=42, shuffle=True
            )
            
            # Build model with trial parameters
            input_shape = (self.sequence_length, X.shape[2])
            output_shape = (self.forecast_horizon, len(self.available_targets))
            
            model = self.build_model_with_params(trial, input_shape, output_shape)
            
            # Training parameters
            batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
            
            # Callbacks
            callbacks = [
                EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=0)
            ]
            
            # Train model
            history = model.fit(
                X_train, y_train,
                epochs=50,  # Reduced for faster optimization
                batch_size=batch_size,
                validation_data=(X_val, y_val),
                callbacks=callbacks,
                verbose=0
            )
            
            # Get best validation loss
            best_val_loss = min(history.history['val_loss'])
            
            # Clean up to free memory
            del model
            tf.keras.backend.clear_session()
            
            return best_val_loss
            
        except Exception as e:
            print(f"Trial failed with error: {e}")
            return float('inf')
    
    def optimize_hyperparameters(self, n_trials=20, timeout=None):
        
        print("\n" + "="*70)
        print("🔧 HYPERPARAMETER OPTIMIZATION WITH OPTUNA")
        print("="*70)
        
        # Create study
        self.study = optuna.create_study(direction='minimize', 
                                        study_name='temperature_lstm_optimization')
        
        print(f"Starting optimization with {n_trials} trials...")
        if timeout:
            print(f"Timeout set to {timeout} seconds")
        
        # Optimize
        self.study.optimize(self.objective, n_trials=n_trials, timeout=timeout)
        
        # Get best parameters
        self.best_params = self.study.best_params
        
        print("\n🏆 OPTIMIZATION COMPLETED!")
        print("="*50)
        print(f"Best validation loss: {self.study.best_value:.6f}")
        print(f"Number of trials: {len(self.study.trials)}")
        
        print(f"\n🎯 BEST HYPERPARAMETERS:")
        print("-" * 40)
        for key, value in self.best_params.items():
            print(f"{key:20s}: {value}")
        
        return self.best_params
    
    def build_best_model(self, input_shape, output_shape):
        """Build model with best parameters found by Optuna"""
        if self.best_params is None:
            raise ValueError("No optimization performed yet. Run optimize_hyperparameters() first.")
        
        model = Sequential()
        
        # Use best parameters
        lstm1_units = self.best_params['lstm1_units']
        lstm2_units = self.best_params['lstm2_units']
        lstm3_units = self.best_params['lstm3_units']
        dense1_units = self.best_params['dense1_units']
        dense2_units = self.best_params['dense2_units']
        dropout_rate = self.best_params['dropout_rate']
        recurrent_dropout = self.best_params['recurrent_dropout']
        l2_reg = self.best_params['l2_reg']
        learning_rate = self.best_params['learning_rate']
        batch_norm = self.best_params['batch_norm']
        
        # Build architecture
        model.add(LSTM(lstm1_units, return_sequences=True, input_shape=input_shape,
                      kernel_regularizer=l2(l2_reg), 
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(LSTM(lstm2_units, return_sequences=True,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(LSTM(lstm3_units, return_sequences=False,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense1_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        if batch_norm:
            model.add(BatchNormalization())
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense2_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        model.add(Dropout(dropout_rate))
        
        model.add(Dense(output_shape[0] * output_shape[1], activation='linear'))
        model.add(tf.keras.layers.Reshape(output_shape))
        
        # Compile with best parameters
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
            loss='mse',
            metrics=['mae']
        )
        
        return model
    
    def train_optimized_model(self, validation_split=0.2, epochs=200):
        
        print("\n" + "="*70)
        print("TRAINING OPTIMIZED TEMPERATURE MAX LSTM MODEL")
        print("="*70)
        
        if self.best_params is None:
            raise ValueError("No optimization performed yet. Run optimize_hyperparameters() first.")
        
        # Create sequences
        X, y = self.create_sequences()
        
        if len(X) == 0:
            raise ValueError("No sequences created!")
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, shuffle=True
        )
        
        print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
        
        # Build optimized model
        input_shape = (self.sequence_length, X.shape[2])
        output_shape = (self.forecast_horizon, len(self.available_targets))
        
        self.model = self.build_best_model(input_shape, output_shape)
        
        print(f"\nOptimized model architecture:")
        self.model.summary()
        
        # Use best batch size
        batch_size = self.best_params.get('batch_size', 32)
        
        # Callbacks
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)
        ]
        
        # Train model
        print(f"\nTraining optimized model...")
        history = self.model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            callbacks=callbacks,
            verbose=1
        )
        
        # Evaluate
        print(f"\nEvaluating optimized model...")
        y_pred = self.model.predict(X_test, verbose=0)
        
        return self.evaluate_model(y_test, y_pred, history)
    
    def evaluate_model(self, y_true, y_pred, history):
        print("\n" + "="*70)
        print("OPTIMIZED TEMPERATURE MAX LSTM RESULTS")
        print("="*70)
        
        # Inverse transform predictions and true values to original scale
        y_true_original = np.zeros_like(y_true)
        y_pred_original = np.zeros_like(y_pred)
        
        for day in range(self.forecast_horizon):
            y_true_original[:, day, :] = self.scaler_targets.inverse_transform(y_true[:, day, :])
            y_pred_original[:, day, :] = self.scaler_targets.inverse_transform(y_pred[:, day, :])
        
        # Calculate metrics for each target variable and forecast day
        results = {}
        
        for i, target_name in enumerate(self.available_targets):
            print(f"\n🌡️ {target_name}:")
            print("-" * 50)
            
            target_results = {
                'mae': [],
                'rmse': [],
                'r2': [],
                'mape': []
            }
            
            for day in range(self.forecast_horizon):
                y_true_day = y_true_original[:, day, i]
                y_pred_day = y_pred_original[:, day, i]
                
                mae = mean_absolute_error(y_true_day, y_pred_day)
                rmse = np.sqrt(mean_squared_error(y_true_day, y_pred_day))
                r2 = r2_score(y_true_day, y_pred_day)
                
                # Calculate MAPE (Mean Absolute Percentage Error)
                mape = np.mean(np.abs((y_true_day - y_pred_day) / (y_true_day + 1e-8))) * 100
                
                target_results['mae'].append(mae)
                target_results['rmse'].append(rmse)
                target_results['r2'].append(r2)
                target_results['mape'].append(mape)
                
                print(f"  Day {day+1}: MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f}, MAPE={mape:.2f}%")
            
            # Calculate averages
            avg_mae = np.mean(target_results['mae'])
            avg_rmse = np.mean(target_results['rmse'])
            avg_r2 = np.mean(target_results['r2'])
            avg_mape = np.mean(target_results['mape'])
            
            target_results['avg_mae'] = avg_mae
            target_results['avg_rmse'] = avg_rmse
            target_results['avg_r2'] = avg_r2
            target_results['avg_mape'] = avg_mape
            
            print(f"  Average: MAE={avg_mae:.3f}, RMSE={avg_rmse:.3f}, R²={avg_r2:.3f}, MAPE={avg_mape:.2f}%")
            
            results[target_name] = target_results
        
        # Overall performance summary
        print(f"\n📊 OVERALL PERFORMANCE SUMMARY:")
        print("="*50)
        
        all_maes = [results[target]['avg_mae'] for target in self.available_targets]
        all_rmses = [results[target]['avg_rmse'] for target in self.available_targets]
        all_r2s = [results[target]['avg_r2'] for target in self.available_targets]
        all_mapes = [results[target]['avg_mape'] for target in self.available_targets]
        
        print(f"Overall Average MAE: {np.mean(all_maes):.3f}")
        print(f"Overall Average RMSE: {np.mean(all_rmses):.3f}")
        print(f"Overall Average R²: {np.mean(all_r2s):.3f}")
        print(f"Overall Average MAPE: {np.mean(all_mapes):.2f}%")
        
        # Optimization summary
        if self.best_params:
            print(f"\n🔧 OPTIMIZATION SUMMARY:")
            print("-" * 40)
            print(f"Best validation loss: {self.study.best_value:.6f}")
            print(f"Number of trials completed: {len(self.study.trials)}")
            print(f"Best LSTM architecture: {self.best_params['lstm1_units']}-{self.best_params['lstm2_units']}-{self.best_params['lstm3_units']}")
            print(f"Best Dense architecture: {self.best_params['dense1_units']}-{self.best_params['dense2_units']}")
            print(f"Best learning rate: {self.best_params['learning_rate']:.6f}")
            print(f"Best dropout rate: {self.best_params['dropout_rate']:.3f}")
        
        return results, history, y_true_original, y_pred_original
    
    
    def predict_future(self, recent_data):
        if isinstance(recent_data, pd.DataFrame):
            recent_scaled = self.scaler_features.transform(recent_data[self.feature_cols])
        elif isinstance(recent_data, np.ndarray):
            if recent_data.shape[1] == len(self.feature_cols):
                recent_scaled = self.scaler_features.transform(recent_data)
            else:
                recent_scaled = recent_data
        else:
            raise ValueError("recent_data must be a DataFrame or NumPy array")

        input_seq = recent_scaled[-self.sequence_length:].reshape(1, self.sequence_length, -1)
        prediction_scaled = self.model.predict(input_seq, verbose=0)

        # Inverse transform
        prediction_original = np.zeros_like(prediction_scaled)
        for day in range(self.forecast_horizon):
            prediction_original[0, day, :] = self.scaler_targets.inverse_transform(
                prediction_scaled[0, day, :].reshape(1, -1)
            )

        predicted_values = prediction_original[0, :, 0]  

        return predicted_values  
    
    def plot_training_history(self, history):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss
        ax1.plot(history.history['loss'], label='Training Loss')
        ax1.plot(history.history['val_loss'], label='Validation Loss')
        ax1.set_title('Optimized Model Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        
        # MAE
        ax2.plot(history.history['mae'], label='Training MAE')
        ax2.plot(history.history['val_mae'], label='Validation MAE')
        ax2.set_title('Optimized Model MAE')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('MAE')
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
    
    def plot_optimization_history(self):
        
        if self.study is None:
            print("No optimization study available to plot.")
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Optimization history
        values = [trial.value for trial in self.study.trials if trial.value is not None]
        ax1.plot(values)
        ax1.set_title('Optimization History')
        ax1.set_xlabel('Trial')
        ax1.set_ylabel('Validation Loss')
        ax1.grid(True)
        
        # Parameter importance (if available)
        try:
            importance = optuna.importance.get_param_importances(self.study)
            params = list(importance.keys())
            importances = list(importance.values())
            
            ax2.barh(params, importances)
            ax2.set_title('Parameter Importance')
            ax2.set_xlabel('Importance')
        except:
            ax2.text(0.5, 0.5, 'Parameter importance\nnot available', 
                    ha='center', va='center', transform=ax2.transAxes)
        
        plt.tight_layout()
        plt.show()

In [5]:
class OptimizedTemperatureMinLSTM:
    def __init__(self, df, sequence_length=10, forecast_horizon=3):
        self.df = df.copy()
        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.scaler_features = None
        self.scaler_targets = None
        self.model = None
        self.best_params = None
        self.study = None
        
        # Temperature target columns
        self.target_cols = [
            'temperature_2m_min (°C)'
        ]
        
    def preprocess_data(self):
        print("🧹 Preprocessing data for max temperature prediction...")
        df = self.df.copy()
        
        # Check which target columns are available
        available_targets = [col for col in self.target_cols if col in df.columns]
        if not available_targets:
            raise ValueError(f"None of the target columns {self.target_cols} found in the dataframe")
        
        self.available_targets = available_targets
        print(f"Available target columns: {available_targets}")
        
        # Handle missing values for targets
        for col in available_targets:
            df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            df[col] = df[col].interpolate(method='linear')
            df[col].fillna(df[col].median(), inplace=True)
        
        # Remove rows with any missing target values
        df = df.dropna(subset=available_targets)
        
        # Define feature columns (exclude date, weather_condition, and target columns)
        exclude_cols = ['date', 'weather_condition', 'relative_humidity_2m_max (%)', 'relative_humidity_2m_min (%)', 'temperature_2m_max (°C)'] + available_targets
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        
        # Handle missing values for features
        numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            df[col] = df[col].interpolate(method='linear')
            df[col].fillna(df[col].median(), inplace=True)
        
        # Scale features
        self.scaler_features = StandardScaler()
        df[feature_cols] = self.scaler_features.fit_transform(df[feature_cols])
        
        # Scale targets
        self.scaler_targets = StandardScaler()
        df[available_targets] = self.scaler_targets.fit_transform(df[available_targets])
        
        # Store processed data
        self.df_processed = df
        self.feature_cols = feature_cols
        
        print(f"Feature columns ({len(feature_cols)}): {feature_cols[:5]}..." if len(feature_cols) > 5 else f"Feature columns: {feature_cols}")
        print(f"Target columns: {available_targets}")
        print(f"Data shape after preprocessing: {df.shape}")
        
        return df
    
    def create_sequences(self):
        print(f"🪟 Creating sequences for Maximum temperature prediction...")
        
        X, y = [], []
        data = self.df_processed
        features = data[self.feature_cols].values
        targets = data[self.available_targets].values
        
        # Create sequences with sliding window
        for i in range(len(data) - self.sequence_length - self.forecast_horizon + 1):
            # Input sequence
            X_seq = features[i:i + self.sequence_length]
            
            # Target sequence (next forecast_horizon days)
            y_seq = targets[i + self.sequence_length:i + self.sequence_length + self.forecast_horizon]
            
            X.append(X_seq)
            y.append(y_seq)
        
        X = np.array(X)
        y = np.array(y)
        
        print(f"Created sequences: X={X.shape}, y={y.shape}")
        print(f"Input sequence length: {self.sequence_length}")
        print(f"Forecast horizon: {self.forecast_horizon}")
        print(f"Number of features: {X.shape[2]}")
        print(f"Number of target variables: {y.shape[2]}")
        
        return X, y
    
    def build_model_with_params(self, trial, input_shape, output_shape):
        
        # Hyperparameters to optimize
        lstm1_units = trial.suggest_int('lstm1_units', 32, 256, step=32)
        lstm2_units = trial.suggest_int('lstm2_units', 16, 128, step=16)
        lstm3_units = trial.suggest_int('lstm3_units', 8, 64, step=8)
        
        dense1_units = trial.suggest_int('dense1_units', 32, 128, step=16)
        dense2_units = trial.suggest_int('dense2_units', 16, 64, step=8)
        
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
        recurrent_dropout = trial.suggest_float('recurrent_dropout', 0.1, 0.4, step=0.1)
        l2_reg = trial.suggest_float('l2_reg', 1e-5, 1e-2, log=True)
        
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        batch_norm = trial.suggest_categorical('batch_norm', [True, False])
        
        model = Sequential()
        
        # First LSTM layer
        model.add(LSTM(lstm1_units, return_sequences=True, input_shape=input_shape,
                      kernel_regularizer=l2(l2_reg), 
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        # Second LSTM layer
        model.add(LSTM(lstm2_units, return_sequences=True,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        # Third LSTM layer
        model.add(LSTM(lstm3_units, return_sequences=False,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(Dropout(dropout_rate + 0.1))
        
        # Dense layers
        model.add(Dense(dense1_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        if batch_norm:
            model.add(BatchNormalization())
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense2_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        model.add(Dropout(dropout_rate))
        
        # Output layer
        model.add(Dense(output_shape[0] * output_shape[1], activation='linear'))
        
        # Reshape to (forecast_horizon, num_target_variables)
        model.add(tf.keras.layers.Reshape(output_shape))
        
        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
            loss='mse',
            metrics=['mae']
        )
        
        return model
    
    def objective(self, trial):
        
        try:
            # Create sequences
            X, y = self.create_sequences()
            
            if len(X) == 0:
                return float('inf')
            
            # Train-validation split
            X_train, X_val, y_train, y_val = train_test_split(
                X, y, test_size=0.2, random_state=42, shuffle=True
            )
            
            # Build model with trial parameters
            input_shape = (self.sequence_length, X.shape[2])
            output_shape = (self.forecast_horizon, len(self.available_targets))
            
            model = self.build_model_with_params(trial, input_shape, output_shape)
            
            # Training parameters
            batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
            
            # Callbacks
            callbacks = [
                EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=0)
            ]
            
            # Train model
            history = model.fit(
                X_train, y_train,
                epochs=50,  # Reduced for faster optimization
                batch_size=batch_size,
                validation_data=(X_val, y_val),
                callbacks=callbacks,
                verbose=0
            )
            
            # Get best validation loss
            best_val_loss = min(history.history['val_loss'])
            
            # Clean up to free memory
            del model
            tf.keras.backend.clear_session()
            
            return best_val_loss
            
        except Exception as e:
            print(f"Trial failed with error: {e}")
            return float('inf')
    
    def optimize_hyperparameters(self, n_trials=20, timeout=None):
        
        print("\n" + "="*70)
        print("🔧 HYPERPARAMETER OPTIMIZATION WITH OPTUNA")
        print("="*70)
        
        # Create study
        self.study = optuna.create_study(direction='minimize', 
                                        study_name='temperature_lstm_optimization')
        
        print(f"Starting optimization with {n_trials} trials...")
        if timeout:
            print(f"Timeout set to {timeout} seconds")
        
        # Optimize
        self.study.optimize(self.objective, n_trials=n_trials, timeout=timeout)
        
        # Get best parameters
        self.best_params = self.study.best_params
        
        print("\n🏆 OPTIMIZATION COMPLETED!")
        print("="*50)
        print(f"Best validation loss: {self.study.best_value:.6f}")
        print(f"Number of trials: {len(self.study.trials)}")
        
        print(f"\n🎯 BEST HYPERPARAMETERS:")
        print("-" * 40)
        for key, value in self.best_params.items():
            print(f"{key:20s}: {value}")
        
        return self.best_params
    
    def build_best_model(self, input_shape, output_shape):
        """Build model with best parameters found by Optuna"""
        if self.best_params is None:
            raise ValueError("No optimization performed yet. Run optimize_hyperparameters() first.")
        
        model = Sequential()
        
        # Use best parameters
        lstm1_units = self.best_params['lstm1_units']
        lstm2_units = self.best_params['lstm2_units']
        lstm3_units = self.best_params['lstm3_units']
        dense1_units = self.best_params['dense1_units']
        dense2_units = self.best_params['dense2_units']
        dropout_rate = self.best_params['dropout_rate']
        recurrent_dropout = self.best_params['recurrent_dropout']
        l2_reg = self.best_params['l2_reg']
        learning_rate = self.best_params['learning_rate']
        batch_norm = self.best_params['batch_norm']
        
        # Build architecture
        model.add(LSTM(lstm1_units, return_sequences=True, input_shape=input_shape,
                      kernel_regularizer=l2(l2_reg), 
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(LSTM(lstm2_units, return_sequences=True,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(LSTM(lstm3_units, return_sequences=False,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense1_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        if batch_norm:
            model.add(BatchNormalization())
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense2_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        model.add(Dropout(dropout_rate))
        
        model.add(Dense(output_shape[0] * output_shape[1], activation='linear'))
        model.add(tf.keras.layers.Reshape(output_shape))
        
        # Compile with best parameters
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
            loss='mse',
            metrics=['mae']
        )
        
        return model
    
    def train_optimized_model(self, validation_split=0.2, epochs=200):
        
        print("\n" + "="*70)
        print("TRAINING OPTIMIZED TEMPERATURE MAX LSTM MODEL")
        print("="*70)
        
        if self.best_params is None:
            raise ValueError("No optimization performed yet. Run optimize_hyperparameters() first.")
        
        # Create sequences
        X, y = self.create_sequences()
        
        if len(X) == 0:
            raise ValueError("No sequences created!")
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, shuffle=True
        )
        
        print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
        
        # Build optimized model
        input_shape = (self.sequence_length, X.shape[2])
        output_shape = (self.forecast_horizon, len(self.available_targets))
        
        self.model = self.build_best_model(input_shape, output_shape)
        
        print(f"\nOptimized model architecture:")
        self.model.summary()
        
        # Use best batch size
        batch_size = self.best_params.get('batch_size', 32)
        
        # Callbacks
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)
        ]
        
        # Train model
        print(f"\nTraining optimized model...")
        history = self.model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            callbacks=callbacks,
            verbose=1
        )
        
        # Evaluate
        print(f"\nEvaluating optimized model...")
        y_pred = self.model.predict(X_test, verbose=0)
        
        return self.evaluate_model(y_test, y_pred, history)
    
    def evaluate_model(self, y_true, y_pred, history):
        print("\n" + "="*70)
        print("OPTIMIZED TEMPERATURE MAX LSTM RESULTS")
        print("="*70)
        
        # Inverse transform predictions and true values to original scale
        y_true_original = np.zeros_like(y_true)
        y_pred_original = np.zeros_like(y_pred)
        
        for day in range(self.forecast_horizon):
            y_true_original[:, day, :] = self.scaler_targets.inverse_transform(y_true[:, day, :])
            y_pred_original[:, day, :] = self.scaler_targets.inverse_transform(y_pred[:, day, :])
        
        # Calculate metrics for each target variable and forecast day
        results = {}
        
        for i, target_name in enumerate(self.available_targets):
            print(f"\n🌡️ {target_name}:")
            print("-" * 50)
            
            target_results = {
                'mae': [],
                'rmse': [],
                'r2': [],
                'mape': []
            }
            
            for day in range(self.forecast_horizon):
                y_true_day = y_true_original[:, day, i]
                y_pred_day = y_pred_original[:, day, i]
                
                mae = mean_absolute_error(y_true_day, y_pred_day)
                rmse = np.sqrt(mean_squared_error(y_true_day, y_pred_day))
                r2 = r2_score(y_true_day, y_pred_day)
                
                # Calculate MAPE (Mean Absolute Percentage Error)
                mape = np.mean(np.abs((y_true_day - y_pred_day) / (y_true_day + 1e-8))) * 100
                
                target_results['mae'].append(mae)
                target_results['rmse'].append(rmse)
                target_results['r2'].append(r2)
                target_results['mape'].append(mape)
                
                print(f"  Day {day+1}: MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f}, MAPE={mape:.2f}%")
            
            # Calculate averages
            avg_mae = np.mean(target_results['mae'])
            avg_rmse = np.mean(target_results['rmse'])
            avg_r2 = np.mean(target_results['r2'])
            avg_mape = np.mean(target_results['mape'])
            
            target_results['avg_mae'] = avg_mae
            target_results['avg_rmse'] = avg_rmse
            target_results['avg_r2'] = avg_r2
            target_results['avg_mape'] = avg_mape
            
            print(f"  Average: MAE={avg_mae:.3f}, RMSE={avg_rmse:.3f}, R²={avg_r2:.3f}, MAPE={avg_mape:.2f}%")
            
            results[target_name] = target_results
        
        # Overall performance summary
        print(f"\n📊 OVERALL PERFORMANCE SUMMARY:")
        print("="*50)
        
        all_maes = [results[target]['avg_mae'] for target in self.available_targets]
        all_rmses = [results[target]['avg_rmse'] for target in self.available_targets]
        all_r2s = [results[target]['avg_r2'] for target in self.available_targets]
        all_mapes = [results[target]['avg_mape'] for target in self.available_targets]
        
        print(f"Overall Average MAE: {np.mean(all_maes):.3f}")
        print(f"Overall Average RMSE: {np.mean(all_rmses):.3f}")
        print(f"Overall Average R²: {np.mean(all_r2s):.3f}")
        print(f"Overall Average MAPE: {np.mean(all_mapes):.2f}%")
        
        # Optimization summary
        if self.best_params:
            print(f"\n🔧 OPTIMIZATION SUMMARY:")
            print("-" * 40)
            print(f"Best validation loss: {self.study.best_value:.6f}")
            print(f"Number of trials completed: {len(self.study.trials)}")
            print(f"Best LSTM architecture: {self.best_params['lstm1_units']}-{self.best_params['lstm2_units']}-{self.best_params['lstm3_units']}")
            print(f"Best Dense architecture: {self.best_params['dense1_units']}-{self.best_params['dense2_units']}")
            print(f"Best learning rate: {self.best_params['learning_rate']:.6f}")
            print(f"Best dropout rate: {self.best_params['dropout_rate']:.3f}")
        
        return results, history, y_true_original, y_pred_original
    
    def predict_future(self, recent_data):
        if isinstance(recent_data, pd.DataFrame):
            recent_scaled = self.scaler_features.transform(recent_data[self.feature_cols])
        elif isinstance(recent_data, np.ndarray):
            if recent_data.shape[1] == len(self.feature_cols):
                recent_scaled = self.scaler_features.transform(recent_data)
            else:
                recent_scaled = recent_data
        else:
            raise ValueError("recent_data must be a DataFrame or NumPy array")

        input_seq = recent_scaled[-self.sequence_length:].reshape(1, self.sequence_length, -1)
        prediction_scaled = self.model.predict(input_seq, verbose=0)

        # Inverse transform
        prediction_original = np.zeros_like(prediction_scaled)
        for day in range(self.forecast_horizon):
            prediction_original[0, day, :] = self.scaler_targets.inverse_transform(
                prediction_scaled[0, day, :].reshape(1, -1)
            )

        predicted_values = prediction_original[0, :, 0] 

        return predicted_values
    
    def plot_training_history(self, history):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss
        ax1.plot(history.history['loss'], label='Training Loss')
        ax1.plot(history.history['val_loss'], label='Validation Loss')
        ax1.set_title('Optimized Model Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        
        # MAE
        ax2.plot(history.history['mae'], label='Training MAE')
        ax2.plot(history.history['val_mae'], label='Validation MAE')
        ax2.set_title('Optimized Model MAE')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('MAE')
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
    
    def plot_optimization_history(self):
        
        if self.study is None:
            print("No optimization study available to plot.")
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Optimization history
        values = [trial.value for trial in self.study.trials if trial.value is not None]
        ax1.plot(values)
        ax1.set_title('Optimization History')
        ax1.set_xlabel('Trial')
        ax1.set_ylabel('Validation Loss')
        ax1.grid(True)
        
        # Parameter importance (if available)
        try:
            importance = optuna.importance.get_param_importances(self.study)
            params = list(importance.keys())
            importances = list(importance.values())
            
            ax2.barh(params, importances)
            ax2.set_title('Parameter Importance')
            ax2.set_xlabel('Importance')
        except:
            ax2.text(0.5, 0.5, 'Parameter importance\nnot available', 
                    ha='center', va='center', transform=ax2.transAxes)
        
        plt.tight_layout()
        plt.show()

In [6]:
class OptimizedTemperatureMaxLSTM:
    def __init__(self, df, sequence_length=10, forecast_horizon=3):
        self.df = df.copy()
        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.scaler_features = None
        self.scaler_targets = None
        self.model = None
        self.best_params = None
        self.study = None
        
        # Temperature target columns
        self.target_cols = [
            'temperature_2m_max (°C)'
        ]
        
    def preprocess_data(self):
        print("🧹 Preprocessing data for max temperature prediction...")
        df = self.df.copy()
        
        # Check which target columns are available
        available_targets = [col for col in self.target_cols if col in df.columns]
        if not available_targets:
            raise ValueError(f"None of the target columns {self.target_cols} found in the dataframe")
        
        self.available_targets = available_targets
        print(f"Available target columns: {available_targets}")
        
        # Handle missing values for targets
        for col in available_targets:
            df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            df[col] = df[col].interpolate(method='linear')
            df[col].fillna(df[col].median(), inplace=True)
        
        # Remove rows with any missing target values
        df = df.dropna(subset=available_targets)
        
        # Define feature columns (exclude date, weather_condition, and target columns)
        exclude_cols = ['date', 'weather_condition', 'relative_humidity_2m_max (%)', 'relative_humidity_2m_min (%)', 'temperature_2m_min (°C)'] + available_targets
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        
        # Handle missing values for features
        numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            df[col] = df[col].interpolate(method='linear')
            df[col].fillna(df[col].median(), inplace=True)
        
        # Scale features
        self.scaler_features = StandardScaler()
        df[feature_cols] = self.scaler_features.fit_transform(df[feature_cols])
        
        # Scale targets
        self.scaler_targets = StandardScaler()
        df[available_targets] = self.scaler_targets.fit_transform(df[available_targets])
        
        # Store processed data
        self.df_processed = df
        self.feature_cols = feature_cols
        
        print(f"Feature columns ({len(feature_cols)}): {feature_cols[:5]}..." if len(feature_cols) > 5 else f"Feature columns: {feature_cols}")
        print(f"Target columns: {available_targets}")
        print(f"Data shape after preprocessing: {df.shape}")
        
        return df
    
    def create_sequences(self):
        print(f"🪟 Creating sequences for Maximum temperature prediction...")
        
        X, y = [], []
        data = self.df_processed
        features = data[self.feature_cols].values
        targets = data[self.available_targets].values
        
        # Create sequences with sliding window
        for i in range(len(data) - self.sequence_length - self.forecast_horizon + 1):
            # Input sequence
            X_seq = features[i:i + self.sequence_length]
            
            # Target sequence (next forecast_horizon days)
            y_seq = targets[i + self.sequence_length:i + self.sequence_length + self.forecast_horizon]
            
            X.append(X_seq)
            y.append(y_seq)
        
        X = np.array(X)
        y = np.array(y)
        
        print(f"Created sequences: X={X.shape}, y={y.shape}")
        print(f"Input sequence length: {self.sequence_length}")
        print(f"Forecast horizon: {self.forecast_horizon}")
        print(f"Number of features: {X.shape[2]}")
        print(f"Number of target variables: {y.shape[2]}")
        
        return X, y
    
    def build_model_with_params(self, trial, input_shape, output_shape):
        
        # Hyperparameters to optimize
        lstm1_units = trial.suggest_int('lstm1_units', 32, 256, step=32)
        lstm2_units = trial.suggest_int('lstm2_units', 16, 128, step=16)
        lstm3_units = trial.suggest_int('lstm3_units', 8, 64, step=8)
        
        dense1_units = trial.suggest_int('dense1_units', 32, 128, step=16)
        dense2_units = trial.suggest_int('dense2_units', 16, 64, step=8)
        
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
        recurrent_dropout = trial.suggest_float('recurrent_dropout', 0.1, 0.4, step=0.1)
        l2_reg = trial.suggest_float('l2_reg', 1e-5, 1e-2, log=True)
        
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        batch_norm = trial.suggest_categorical('batch_norm', [True, False])
        
        model = Sequential()
        
        # First LSTM layer
        model.add(LSTM(lstm1_units, return_sequences=True, input_shape=input_shape,
                      kernel_regularizer=l2(l2_reg), 
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        # Second LSTM layer
        model.add(LSTM(lstm2_units, return_sequences=True,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        # Third LSTM layer
        model.add(LSTM(lstm3_units, return_sequences=False,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(Dropout(dropout_rate + 0.1))
        
        # Dense layers
        model.add(Dense(dense1_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        if batch_norm:
            model.add(BatchNormalization())
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense2_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        model.add(Dropout(dropout_rate))
        
        # Output layer
        model.add(Dense(output_shape[0] * output_shape[1], activation='linear'))
        
        # Reshape to (forecast_horizon, num_target_variables)
        model.add(tf.keras.layers.Reshape(output_shape))
        
        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
            loss='mse',
            metrics=['mae']
        )
        
        return model
    
    def objective(self, trial):
        
        try:
            # Create sequences
            X, y = self.create_sequences()
            
            if len(X) == 0:
                return float('inf')
            
            # Train-validation split
            X_train, X_val, y_train, y_val = train_test_split(
                X, y, test_size=0.2, random_state=42, shuffle=True
            )
            
            # Build model with trial parameters
            input_shape = (self.sequence_length, X.shape[2])
            output_shape = (self.forecast_horizon, len(self.available_targets))
            
            model = self.build_model_with_params(trial, input_shape, output_shape)
            
            # Training parameters
            batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
            
            # Callbacks
            callbacks = [
                EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=0)
            ]
            
            # Train model
            history = model.fit(
                X_train, y_train,
                epochs=50,  # Reduced for faster optimization
                batch_size=batch_size,
                validation_data=(X_val, y_val),
                callbacks=callbacks,
                verbose=0
            )
            
            # Get best validation loss
            best_val_loss = min(history.history['val_loss'])
            
            # Clean up to free memory
            del model
            tf.keras.backend.clear_session()
            
            return best_val_loss
            
        except Exception as e:
            print(f"Trial failed with error: {e}")
            return float('inf')
    
    def optimize_hyperparameters(self, n_trials=20, timeout=None):
        
        print("\n" + "="*70)
        print("🔧 HYPERPARAMETER OPTIMIZATION WITH OPTUNA")
        print("="*70)
        
        # Create study
        self.study = optuna.create_study(direction='minimize', 
                                        study_name='temperature_lstm_optimization')
        
        print(f"Starting optimization with {n_trials} trials...")
        if timeout:
            print(f"Timeout set to {timeout} seconds")
        
        # Optimize
        self.study.optimize(self.objective, n_trials=n_trials, timeout=timeout)
        
        # Get best parameters
        self.best_params = self.study.best_params
        
        print("\n🏆 OPTIMIZATION COMPLETED!")
        print("="*50)
        print(f"Best validation loss: {self.study.best_value:.6f}")
        print(f"Number of trials: {len(self.study.trials)}")
        
        print(f"\n🎯 BEST HYPERPARAMETERS:")
        print("-" * 40)
        for key, value in self.best_params.items():
            print(f"{key:20s}: {value}")
        
        return self.best_params
    
    def build_best_model(self, input_shape, output_shape):
        """Build model with best parameters found by Optuna"""
        if self.best_params is None:
            raise ValueError("No optimization performed yet. Run optimize_hyperparameters() first.")
        
        model = Sequential()
        
        # Use best parameters
        lstm1_units = self.best_params['lstm1_units']
        lstm2_units = self.best_params['lstm2_units']
        lstm3_units = self.best_params['lstm3_units']
        dense1_units = self.best_params['dense1_units']
        dense2_units = self.best_params['dense2_units']
        dropout_rate = self.best_params['dropout_rate']
        recurrent_dropout = self.best_params['recurrent_dropout']
        l2_reg = self.best_params['l2_reg']
        learning_rate = self.best_params['learning_rate']
        batch_norm = self.best_params['batch_norm']
        
        # Build architecture
        model.add(LSTM(lstm1_units, return_sequences=True, input_shape=input_shape,
                      kernel_regularizer=l2(l2_reg), 
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(LSTM(lstm2_units, return_sequences=True,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(LSTM(lstm3_units, return_sequences=False,
                      kernel_regularizer=l2(l2_reg),
                      dropout=dropout_rate, recurrent_dropout=recurrent_dropout))
        if batch_norm:
            model.add(BatchNormalization())
        
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense1_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        if batch_norm:
            model.add(BatchNormalization())
        model.add(Dropout(dropout_rate + 0.1))
        
        model.add(Dense(dense2_units, activation='relu', kernel_regularizer=l2(l2_reg)))
        model.add(Dropout(dropout_rate))
        
        model.add(Dense(output_shape[0] * output_shape[1], activation='linear'))
        model.add(tf.keras.layers.Reshape(output_shape))
        
        # Compile with best parameters
        model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
            loss='mse',
            metrics=['mae']
        )
        
        return model
    
    def train_optimized_model(self, validation_split=0.2, epochs=200):
        
        print("\n" + "="*70)
        print("TRAINING OPTIMIZED TEMPERATURE MAX LSTM MODEL")
        print("="*70)
        
        if self.best_params is None:
            raise ValueError("No optimization performed yet. Run optimize_hyperparameters() first.")
        
        # Create sequences
        X, y = self.create_sequences()
        
        if len(X) == 0:
            raise ValueError("No sequences created!")
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, shuffle=True
        )
        
        print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
        
        # Build optimized model
        input_shape = (self.sequence_length, X.shape[2])
        output_shape = (self.forecast_horizon, len(self.available_targets))
        
        self.model = self.build_best_model(input_shape, output_shape)
        
        print(f"\nOptimized model architecture:")
        self.model.summary()
        
        # Use best batch size
        batch_size = self.best_params.get('batch_size', 32)
        
        # Callbacks
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)
        ]
        
        # Train model
        print(f"\nTraining optimized model...")
        history = self.model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            callbacks=callbacks,
            verbose=1
        )
        
        # Evaluate
        print(f"\nEvaluating optimized model...")
        y_pred = self.model.predict(X_test, verbose=0)
        
        return self.evaluate_model(y_test, y_pred, history)
    
    def evaluate_model(self, y_true, y_pred, history):
        print("\n" + "="*70)
        print("OPTIMIZED TEMPERATURE MAX LSTM RESULTS")
        print("="*70)
        
        # Inverse transform predictions and true values to original scale
        y_true_original = np.zeros_like(y_true)
        y_pred_original = np.zeros_like(y_pred)
        
        for day in range(self.forecast_horizon):
            y_true_original[:, day, :] = self.scaler_targets.inverse_transform(y_true[:, day, :])
            y_pred_original[:, day, :] = self.scaler_targets.inverse_transform(y_pred[:, day, :])
        
        # Calculate metrics for each target variable and forecast day
        results = {}
        
        for i, target_name in enumerate(self.available_targets):
            print(f"\n🌡️ {target_name}:")
            print("-" * 50)
            
            target_results = {
                'mae': [],
                'rmse': [],
                'r2': [],
                'mape': []
            }
            
            for day in range(self.forecast_horizon):
                y_true_day = y_true_original[:, day, i]
                y_pred_day = y_pred_original[:, day, i]
                
                mae = mean_absolute_error(y_true_day, y_pred_day)
                rmse = np.sqrt(mean_squared_error(y_true_day, y_pred_day))
                r2 = r2_score(y_true_day, y_pred_day)
                
                # Calculate MAPE (Mean Absolute Percentage Error)
                mape = np.mean(np.abs((y_true_day - y_pred_day) / (y_true_day + 1e-8))) * 100
                
                target_results['mae'].append(mae)
                target_results['rmse'].append(rmse)
                target_results['r2'].append(r2)
                target_results['mape'].append(mape)
                
                print(f"  Day {day+1}: MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f}, MAPE={mape:.2f}%")
            
            # Calculate averages
            avg_mae = np.mean(target_results['mae'])
            avg_rmse = np.mean(target_results['rmse'])
            avg_r2 = np.mean(target_results['r2'])
            avg_mape = np.mean(target_results['mape'])
            
            target_results['avg_mae'] = avg_mae
            target_results['avg_rmse'] = avg_rmse
            target_results['avg_r2'] = avg_r2
            target_results['avg_mape'] = avg_mape
            
            print(f"  Average: MAE={avg_mae:.3f}, RMSE={avg_rmse:.3f}, R²={avg_r2:.3f}, MAPE={avg_mape:.2f}%")
            
            results[target_name] = target_results
        
        # Overall performance summary
        print(f"\n📊 OVERALL PERFORMANCE SUMMARY:")
        print("="*50)
        
        all_maes = [results[target]['avg_mae'] for target in self.available_targets]
        all_rmses = [results[target]['avg_rmse'] for target in self.available_targets]
        all_r2s = [results[target]['avg_r2'] for target in self.available_targets]
        all_mapes = [results[target]['avg_mape'] for target in self.available_targets]
        
        print(f"Overall Average MAE: {np.mean(all_maes):.3f}")
        print(f"Overall Average RMSE: {np.mean(all_rmses):.3f}")
        print(f"Overall Average R²: {np.mean(all_r2s):.3f}")
        print(f"Overall Average MAPE: {np.mean(all_mapes):.2f}%")
        
        # Optimization summary
        if self.best_params:
            print(f"\n🔧 OPTIMIZATION SUMMARY:")
            print("-" * 40)
            print(f"Best validation loss: {self.study.best_value:.6f}")
            print(f"Number of trials completed: {len(self.study.trials)}")
            print(f"Best LSTM architecture: {self.best_params['lstm1_units']}-{self.best_params['lstm2_units']}-{self.best_params['lstm3_units']}")
            print(f"Best Dense architecture: {self.best_params['dense1_units']}-{self.best_params['dense2_units']}")
            print(f"Best learning rate: {self.best_params['learning_rate']:.6f}")
            print(f"Best dropout rate: {self.best_params['dropout_rate']:.3f}")
        
        return results, history, y_true_original, y_pred_original
    
    def predict_future(self, recent_data):
        if isinstance(recent_data, pd.DataFrame):
            recent_scaled = self.scaler_features.transform(recent_data[self.feature_cols])
        elif isinstance(recent_data, np.ndarray):
            if recent_data.shape[1] == len(self.feature_cols):
                recent_scaled = self.scaler_features.transform(recent_data)
            else:
                recent_scaled = recent_data
        else:
            raise ValueError("recent_data must be a DataFrame or NumPy array")

        input_seq = recent_scaled[-self.sequence_length:].reshape(1, self.sequence_length, -1)
        prediction_scaled = self.model.predict(input_seq, verbose=0)

        # Inverse transform
        prediction_original = np.zeros_like(prediction_scaled)
        for day in range(self.forecast_horizon):
            prediction_original[0, day, :] = self.scaler_targets.inverse_transform(
                prediction_scaled[0, day, :].reshape(1, -1)
            )

        predicted_values = prediction_original[0, :, 0]  # shape: (forecast_horizon,)

        return predicted_values

    def plot_training_history(self, history):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss
        ax1.plot(history.history['loss'], label='Training Loss')
        ax1.plot(history.history['val_loss'], label='Validation Loss')
        ax1.set_title('Optimized Model Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        
        # MAE
        ax2.plot(history.history['mae'], label='Training MAE')
        ax2.plot(history.history['val_mae'], label='Validation MAE')
        ax2.set_title('Optimized Model MAE')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('MAE')
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
    
    def plot_optimization_history(self):
        
        if self.study is None:
            print("No optimization study available to plot.")
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Optimization history
        values = [trial.value for trial in self.study.trials if trial.value is not None]
        ax1.plot(values)
        ax1.set_title('Optimization History')
        ax1.set_xlabel('Trial')
        ax1.set_ylabel('Validation Loss')
        ax1.grid(True)
        
        # Parameter importance (if available)
        try:
            importance = optuna.importance.get_param_importances(self.study)
            params = list(importance.keys())
            importances = list(importance.values())
            
            ax2.barh(params, importances)
            ax2.set_title('Parameter Importance')
            ax2.set_xlabel('Importance')
        except:
            ax2.text(0.5, 0.5, 'Parameter importance\nnot available', 
                    ha='center', va='center', transform=ax2.transAxes)
        
        plt.tight_layout()
        plt.show()

In [7]:
class WeatherFeatureEngineer:
    def __init__(self):
        self.cols_to_drop = [
            'dew_point_2m_max (°C)',
            'wind_speed_10m_max (km/h)',
            'surface_pressure_max (hPa)',
            'surface_pressure_min (hPa)',
            'wet_bulb_temperature_2m_max (°C)',
            'wet_bulb_temperature_2m_min (°C)',
            'soil_temperature_0_to_100cm_mean (°C)'
        ]
        self.low_importance_features = [
            'et0_fao_evapotranspiration_sum (mm)'
        ]
        self.desired_order = [
            'date', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'dayofyear', 'month_sin', 'month_cos',
            'dayofyear_sin', 'dayofyear_cos',

            'temperature_2m_max (°C)', 'temperature_2m_min (°C)', 'temp_range',
            'dew_point_2m_max (°C)', 'dew_point_2m_min (°C)', 'dew_point_range',
            'wet_bulb_temperature_2m_max (°C)', 'wet_bulb_temperature_2m_min (°C)',

            'relative_humidity_2m_max (%)', 'relative_humidity_2m_min (%)', 'humidity_range',

            'daylight_duration (s)', 'sunshine_duration (s)', 'sunshine_ratio', 'daylight_to_sunshine_ratio',

            'cloud_cover_min (%)', 'cloud_cover_max (%)',

            'rain_sum (mm)', 'precipitation_hours (h)', 'rain_today',

            'wind_speed_10m_max (km/h)', 'wind_speed_10m_min (km/h)', 'avg_wind_speed',
            'wind_gusts_10m_max (km/h)', 'wind_gusts_10m_min (km/h)', 'wind_gust_range', 'wind_variability',

            'surface_pressure_max (hPa)', 'surface_pressure_min (hPa)',
            'pressure_msl_max (hPa)', 'pressure_msl_min (hPa)', 'pressure_range',

            'soil_moisture_0_to_100cm_mean (m³/m³)', 'soil_temperature_0_to_100cm_mean (°C)',

            'et0_fao_evapotranspiration_sum (mm)',

            'weather_condition'  # target at the end
        ]

    def create_temporal_features(self, df):
        """Create temporal features ensuring all expected features are present"""
        # Ensure date column exists and is properly formatted
        if 'date' not in df.columns:
            # If no date column, create one with current date as starting point
            print("Warning: No 'date' column found. Creating synthetic dates.")
            df['date'] = pd.date_range(start='2024-01-01', periods=len(df), freq='D')
        
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        
        # Handle any NaT values in date column
        if df['date'].isna().any():
            print("Warning: Found NaT values in date column. Forward filling...")
            df['date'] = df['date'].fillna(method='ffill')
            if df['date'].isna().any():  # If still NaT after ffill
                df['date'] = df['date'].fillna(pd.Timestamp('2024-01-01'))
        
        # Basic calendar features
        df['day'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['year'] = df['date'].dt.year
        df['dayofweek'] = df['date'].dt.dayofweek       # Monday=0, Sunday=6
        df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)  # Ensure it's int, not bool
        
        # Seasonal encodings (sin/cos to handle cyclic nature)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['dayofyear'] = df['date'].dt.dayofyear
        df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
        df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
        
        # Debug: Print to verify is_weekend is created
        print(f"DEBUG: is_weekend created. Shape: {df['is_weekend'].shape}, dtype: {df['is_weekend'].dtype}")
        print(f"DEBUG: is_weekend values: {df['is_weekend'].unique()}")
        
        return df

    def create_additional_features(self, df):
        """Create additional features with proper error handling"""
        # Temperature-related features
        if 'temperature_2m_max (°C)' in df.columns and 'temperature_2m_min (°C)' in df.columns:
            df['temp_range'] = df['temperature_2m_max (°C)'] - df['temperature_2m_min (°C)']

        # Wind-related features
        if 'wind_gusts_10m_max (km/h)' in df.columns and 'wind_gusts_10m_min (km/h)' in df.columns:
            df['wind_gust_range'] = df['wind_gusts_10m_max (km/h)'] - df['wind_gusts_10m_min (km/h)']
        
        if 'wind_speed_10m_max (km/h)' in df.columns and 'wind_speed_10m_min (km/h)' in df.columns:
            df['avg_wind_speed'] = (df['wind_speed_10m_max (km/h)'] + df['wind_speed_10m_min (km/h)']) / 2
            df['wind_variability'] = df['wind_speed_10m_max (km/h)'] - df['wind_speed_10m_min (km/h)']

        # Humidity & Dew Point
        if 'relative_humidity_2m_max (%)' in df.columns and 'relative_humidity_2m_min (%)' in df.columns:
            df['humidity_range'] = df['relative_humidity_2m_max (%)'] - df['relative_humidity_2m_min (%)']
        
        if 'dew_point_2m_max (°C)' in df.columns and 'dew_point_2m_min (°C)' in df.columns:
            df['dew_point_range'] = df['dew_point_2m_max (°C)'] - df['dew_point_2m_min (°C)']

        # Cloud & Sun
        if 'sunshine_duration (s)' in df.columns and 'daylight_duration (s)' in df.columns:
            # Avoid division by zero
            df['sunshine_ratio'] = df['sunshine_duration (s)'] / (df['daylight_duration (s)'] + 1e-8)

        # Rain/Precipitation
        if 'rain_sum (mm)' in df.columns:
            df['rain_today'] = df['rain_sum (mm)'].apply(lambda x: 1 if x > 0 else 0)

        # Pressure Features
        if 'surface_pressure_max (hPa)' in df.columns and 'surface_pressure_min (hPa)' in df.columns:
            df['pressure_range'] = df['surface_pressure_max (hPa)'] - df['surface_pressure_min (hPa)']

        # Daylight to Sunshine Ratio
        if 'daylight_duration (s)' in df.columns and 'sunshine_duration (s)' in df.columns:
            df['daylight_to_sunshine_ratio'] = df['daylight_duration (s)'] / (df['sunshine_duration (s)'] + 1)

        # Fill missing values after creating new features
        df = df.fillna(method='ffill').fillna(method='bfill')
        
        # If still NaN values, fill with median for numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if df[col].isna().any():
                df[col] = df[col].fillna(df[col].median())

        return df

    def create_targeted_weather_features(self, df):
        """Create targeted weather features with proper feature preservation"""
        df_enhanced = df.copy()
    
        # Ensure date is datetime for proper sorting
        if 'date' in df_enhanced.columns:
            df_enhanced['date'] = pd.to_datetime(df_enhanced['date'])
            df_enhanced = df_enhanced.sort_values('date').reset_index(drop=True)
        
        print("Creating targeted weather features based on importance analysis...")
        
        # CRITICAL: Check if is_weekend exists before proceeding
        if 'is_weekend' not in df_enhanced.columns:
            print("ERROR: is_weekend not found! Recreating temporal features...")
            df_enhanced = self.create_temporal_features(df_enhanced)
        
        print("1. Engineering Tier 1 (Core Weather State) features...")
        
        tier1_features = [
            'cloud_cover_max (%)',
            'rain_sum (mm)', 
            'precipitation_hours (h)',
            'cloud_cover_min (%)'
        ]
    
        # Focus heavy engineering on these top features
        for feature in tier1_features:
            if feature in df_enhanced.columns:
                feature_clean = feature.replace(' ', '_').replace('(%)', '').replace('(mm)', '').replace('(h)', '')
                
                # Lagged features (1-7 days) - weather persistence is crucial
                for lag in [1, 2, 3, 7]:
                    df_enhanced[f'{feature_clean}_lag_{lag}'] = df_enhanced[feature].shift(lag)
                
                # Rolling statistics (key for weather patterns)
                for window in [3, 7, 14]:
                    df_enhanced[f'{feature_clean}_rolling_mean_{window}d'] = df_enhanced[feature].rolling(window, min_periods=1).mean()
                    df_enhanced[f'{feature_clean}_rolling_std_{window}d'] = df_enhanced[feature].rolling(window, min_periods=1).std()
                    df_enhanced[f'{feature_clean}_rolling_max_{window}d'] = df_enhanced[feature].rolling(window, min_periods=1).max()
                
                # Trend features
                df_enhanced[f'{feature_clean}_3day_trend'] = df_enhanced[feature].diff(3)
                df_enhanced[f'{feature_clean}_7day_trend'] = df_enhanced[feature].diff(7)
                
                # Exponential weighted moving average
                df_enhanced[f'{feature_clean}_ewm_3d'] = df_enhanced[feature].ewm(span=3, min_periods=1).mean()
                df_enhanced[f'{feature_clean}_ewm_7d'] = df_enhanced[feature].ewm(span=7, min_periods=1).mean()
        
        # Special handling for rain_today (binary feature)
        if 'rain_today' in df_enhanced.columns:
            # Rain persistence patterns
            df_enhanced['rain_today_lag_1'] = df_enhanced['rain_today'].shift(1)
            df_enhanced['rain_today_lag_2'] = df_enhanced['rain_today'].shift(2)
            df_enhanced['rain_today_lag_3'] = df_enhanced['rain_today'].shift(3)
            
            # Rolling rain frequency
            df_enhanced['rain_frequency_7d'] = df_enhanced['rain_today'].rolling(7, min_periods=1).mean()
            df_enhanced['rain_frequency_14d'] = df_enhanced['rain_today'].rolling(14, min_periods=1).mean()
            
            # Consecutive rain days
            df_enhanced['consecutive_rain_days'] = (
                df_enhanced['rain_today'].groupby(
                    (df_enhanced['rain_today'] != df_enhanced['rain_today'].shift()).cumsum()
                ).cumsum()
            )
        
            # Days since last rain
            rain_dates = df_enhanced[df_enhanced['rain_today'] == 1].index
            df_enhanced['days_since_rain'] = np.nan
            for i in df_enhanced.index:
                prev_rain = rain_dates[rain_dates < i]
                if len(prev_rain) > 0:
                    df_enhanced.loc[i, 'days_since_rain'] = i - prev_rain.max()
            df_enhanced['days_since_rain'] = df_enhanced['days_since_rain'].fillna(30)  # Cap at 30 days
        
        print("2. Engineering Tier 2 (Physical Variables) features...")
        
        tier2_features = [
            'temp_range',
            'relative_humidity_2m_min (%)',
            'temperature_2m_max (°C)',
            'humidity_range',
            'relative_humidity_2m_max (%)'
        ]
        
        # Lighter engineering for these features
        for feature in tier2_features:
            if feature in df_enhanced.columns:
                feature_clean = feature.replace(' ', '_').replace('(%)', '').replace('(°C)', '')
                
                # Key lagged features only
                for lag in [1, 3, 7]:
                    df_enhanced[f'{feature_clean}_lag_{lag}'] = df_enhanced[feature].shift(lag)
                
                # Essential rolling statistics
                df_enhanced[f'{feature_clean}_rolling_mean_7d'] = df_enhanced[feature].rolling(7, min_periods=1).mean()
                df_enhanced[f'{feature_clean}_rolling_std_7d'] = df_enhanced[feature].rolling(7, min_periods=1).std()
                
                # Trend features
                df_enhanced[f'{feature_clean}_7day_trend'] = df_enhanced[feature].diff(7)
        
        print("3. Creating interaction features...")
        
        # High-impact feature interactions
        if 'cloud_cover_max (%)' in df_enhanced.columns and 'rain_sum (mm)' in df_enhanced.columns:
            df_enhanced['cloud_rain_interaction'] = df_enhanced['cloud_cover_max (%)'] * df_enhanced['rain_sum (mm)']
        
        if 'temp_range' in df_enhanced.columns and 'relative_humidity_2m_min (%)' in df_enhanced.columns:
            df_enhanced['temp_humidity_interaction'] = df_enhanced['temp_range'] * df_enhanced['relative_humidity_2m_min (%)']
        
        if 'precipitation_hours (h)' in df_enhanced.columns and 'cloud_cover_max (%)' in df_enhanced.columns:
            df_enhanced['precip_cloud_interaction'] = df_enhanced['precipitation_hours (h)'] * df_enhanced['cloud_cover_max (%)']
        
        print("4. Creating weather pattern features...")
        
        # Weather stability indicators
        if 'cloud_cover_max (%)' in df_enhanced.columns:
            df_enhanced['cloud_volatility_7d'] = df_enhanced['cloud_cover_max (%)'].rolling(7, min_periods=1).std()
        
        if 'rain_sum (mm)' in df_enhanced.columns:
            df_enhanced['rain_volatility_7d'] = df_enhanced['rain_sum (mm)'].rolling(7, min_periods=1).std()
        
        # Seasonal anomalies for top features
        if 'temperature_2m_max (°C)' in df_enhanced.columns and 'dayofyear' in df_enhanced.columns:
            df_enhanced['temp_seasonal_anomaly'] = (
                df_enhanced['temperature_2m_max (°C)'] - 
                df_enhanced.groupby('dayofyear')['temperature_2m_max (°C)'].transform('mean')
            )
        
        print("5. Optimizing feature set...")
    
        # Identify and remove low-importance features (keeping time features)
        low_importance_features = [
            'et0_fao_evapotranspiration_sum (mm)',  # Remove if score < 0.1 and not in top tier
        ]
        
        # Remove low-importance features (but keep ALL time features)
        time_related_keywords = ['year', 'month', 'day', 'dayof', 'weekend', 'sin', 'cos']
    
        features_to_drop = []
        for feature in low_importance_features:
            if feature in df_enhanced.columns:
                # Don't drop if it's a time-related feature
                if not any(keyword in feature.lower() for keyword in time_related_keywords):
                    features_to_drop.append(feature)
        
        if features_to_drop:
            print(f"Removing {len(features_to_drop)} low-importance non-temporal features...")
            df_enhanced = df_enhanced.drop(columns=features_to_drop)
        
        # CRITICAL: Final check for is_weekend
        if 'is_weekend' not in df_enhanced.columns:
            print("CRITICAL ERROR: is_weekend missing after feature engineering!")
            # Recreate it if missing
            if 'date' in df_enhanced.columns:
                df_enhanced['date'] = pd.to_datetime(df_enhanced['date'])
                df_enhanced['dayofweek'] = df_enhanced['date'].dt.dayofweek
                df_enhanced['is_weekend'] = (df_enhanced['dayofweek'] >= 5).astype(int)
                print("is_weekend recreated!")
        
        original_features = len(df.columns)
        final_features = len(df_enhanced.columns)
        new_features = final_features - original_features
            
        print(f"\n{'='*50}")
        print("TARGETED FEATURE ENGINEERING SUMMARY")
        print(f"{'='*50}")
        print(f"Original features: {original_features}")
        print(f"Final features: {final_features}")
        print(f"New features added: {new_features}")
        print(f"Features removed: {len(features_to_drop)}")
        
        # Feature breakdown
        tier1_count = len([col for col in df_enhanced.columns if any(t1 in col for t1 in ['cloud_cover', 'rain_', 'precipitation'])])
        tier2_count = len([col for col in df_enhanced.columns if any(t2 in col for t2 in ['temp_', 'humidity_'])])
        time_count = len([col for col in df_enhanced.columns if any(t in col.lower() for t in time_related_keywords)])
        
        print(f"\nFeature Categories:")
        print(f"  Weather State (Tier 1): ~{tier1_count} features")
        print(f"  Physical Variables (Tier 2): ~{tier2_count} features") 
        print(f"  Temporal Features: {time_count} features")
        print(f"  Other Features: {final_features - tier1_count - tier2_count - time_count} features")
        
        # Final verification of critical features
        critical_features = ['is_weekend', 'year', 'month', 'day', 'dayofweek']
        missing_critical = [f for f in critical_features if f not in df_enhanced.columns]
        if missing_critical:
            print(f"WARNING: Missing critical features: {missing_critical}")
        else:
            print("✅ All critical temporal features present")
        
        return df_enhanced

    def drop_unimportant_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drop unimportant columns while preserving critical features"""
        # Don't drop temporal features even if they're in cols_to_drop
        temporal_features = ['year', 'month', 'day', 'dayofweek', 'is_weekend', 'dayofyear', 
                           'month_sin', 'month_cos', 'dayofyear_sin', 'dayofyear_cos']
        
        cols_to_drop_safe = [col for col in self.cols_to_drop 
                           if col not in temporal_features and col in df.columns]
        
        if cols_to_drop_safe:
            print(f"Dropping {len(cols_to_drop_safe)} unimportant columns: {cols_to_drop_safe}")
            df = df.drop(columns=cols_to_drop_safe)
        
        return df

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Main transformation pipeline with feature preservation"""
        print("Starting feature transformation pipeline...")
        
        # 🧭 Map weather_code to human-readable weather_condition
        weather_conditions = {
            0: "Clear Sky",
            1: "Partly Clear",
            2: "Partly Clear", 
            3: "Cloudy",
            45: "Cloudy",
            48: "Cloudy",
            51: "Light Drizzle",
            53: "Moderate Drizzle",
            55: "Heavy Drizzle",
            61: "Light Rain",
            63: "Moderate Rain",
            65: "Heavy Rain",
            71: "Light Rain",  # Treat snow as rain for consistency
            73: "Moderate Rain",
            75: "Heavy Rain",
            95: "Heavy Rain"  # Thunderstorm as heavy rain
        }
        
        if 'weather_code' in df.columns:
            df['weather_condition'] = df['weather_code'].map(weather_conditions)
            df = df.drop(columns=['weather_code'], errors='ignore')
        
        # Create temporal features first
        df = self.create_temporal_features(df)
        
        # Verify is_weekend exists
        if 'is_weekend' not in df.columns:
            raise ValueError("Critical error: is_weekend not created in temporal features!")
        
        # Create additional features
        df = self.create_additional_features(df)
        
        # Reorder columns safely (ignore missing columns)
        cols_in_df = [col for col in self.desired_order if col in df.columns]
        remaining_cols = [col for col in df.columns if col not in cols_in_df]
        df = df[cols_in_df + remaining_cols]  # Add any new columns at the end
        
        # Drop unimportant columns (but preserve temporal features)
        df = self.drop_unimportant_columns(df)
        
        # Final verification before advanced feature engineering
        if 'is_weekend' not in df.columns:
            raise ValueError("Critical error: is_weekend lost during basic transformations!")
        
        # Create targeted weather features
        df = self.create_targeted_weather_features(df)
        
        # Final check
        if 'is_weekend' not in df.columns:
            raise ValueError("Critical error: is_weekend lost during targeted feature engineering!")
        
        print("✅ Feature transformation completed successfully!")
        print(f"Final dataframe shape: {df.shape}")
        print(f"is_weekend present: {'is_weekend' in df.columns}")
        
        return df

    def generate_model_inputs(self, df: pd.DataFrame, model_type: str, scaler_X=None):
        """Generate model inputs with proper feature handling"""
        df = df.copy()
        
        # Ensure is_weekend exists
        if 'is_weekend' not in df.columns:
            print("WARNING: is_weekend missing in generate_model_inputs. Recreating...")
            if 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'])
                df['dayofweek'] = df['date'].dt.dayofweek
                df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
            else:
                # If no date, assume weekday
                df['is_weekend'] = 0

        if model_type == 'weather_condition':
            # Classification model handling
            df['weather_condition'] = df['weather_condition'].replace({
                'Mainly Clear': 'Partly Clear',
                'Partly Cloudy': 'Partly Clear'
            })

            df = df.dropna(subset=['weather_condition'])

            # Fill missing numerics
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            for col in numeric_cols:
                df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
                df[col] = df[col].interpolate().fillna(df[col].median())

            # Encode labels
            from sklearn.preprocessing import LabelEncoder
            from sklearn.utils.class_weight import compute_class_weight
            
            label_encoder = LabelEncoder()
            encoded_col = 'weather_condition_encoded'
            df[encoded_col] = label_encoder.fit_transform(df['weather_condition'])

            # Compute class weights
            unique_classes = df[encoded_col].unique()
            class_weights = compute_class_weight('balanced', classes=unique_classes, y=df[encoded_col])
            class_weight_dict = dict(zip(unique_classes, class_weights))

            # Select features (exclude date + original & encoded target)
            feature_cols = [col for col in df.columns if col not in ['date', 'weather_condition', encoded_col]]

            # Scale features
            from sklearn.preprocessing import StandardScaler
            scaler_X = StandardScaler()
            df[feature_cols] = scaler_X.fit_transform(df[feature_cols])

            X = df[feature_cols].values
            y = df[encoded_col].values

            return X, y, feature_cols, label_encoder, class_weight_dict, scaler_X

        else:
            # Regression models
            model_exclude_map = {
                'temp_max': ['date', 'weather_condition', 'relative_humidity_2m_max (%)',
                            'relative_humidity_2m_min (%)', 'temperature_2m_min (°C)', 'temperature_2m_max (°C)'],
                
                'temp_min': ['date', 'weather_condition', 'temperature_2m_max (°C)', 
                            'relative_humidity_2m_max (%)', 'relative_humidity_2m_min (%)', 'temperature_2m_min (°C)'],
                
                'humidity_max': ['date', 'weather_condition', 'temperature_2m_max (°C)', 'temperature_2m_min (°C)',
                                'relative_humidity_2m_min (%)', 'relative_humidity_2m_max (%)'],
                
                'humidity_min': ['date', 'weather_condition', 'temperature_2m_max (°C)', 'temperature_2m_min (°C)',
                                'relative_humidity_2m_min (%)', 'relative_humidity_2m_max (%)']
            }

            if model_type not in model_exclude_map:
                raise ValueError(f"Unknown model_type: {model_type}")

            exclude_cols = model_exclude_map[model_type]

            # Select features
            feature_cols = [col for col in df.columns if col not in exclude_cols]
            numeric_feature_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

            # Verify is_weekend is in the feature set
            if 'is_weekend' not in numeric_feature_cols:
                if 'is_weekend' in df.columns:
                    print(f"WARNING: is_weekend exists but not in numeric_feature_cols. Adding it...")
                    numeric_feature_cols.append('is_weekend')
                else:
                    print(f"ERROR: is_weekend missing for model {model_type}!")
                    raise ValueError(f"is_weekend feature missing for model {model_type}")

            # Clean missing values
            df[numeric_feature_cols] = (
                df[numeric_feature_cols]
                .fillna(method='ffill')
                .fillna(method='bfill')
                .interpolate()
                .fillna(df[numeric_feature_cols].median())
            )

            # Scale if scaler provided
            if scaler_X:
                X_scaled = scaler_X.transform(df[numeric_feature_cols])
                return X_scaled, numeric_feature_cols
            else:
                return df[numeric_feature_cols].values, numeric_feature_cols

In [8]:
@register_keras_serializable()
def focal_loss_fn(y_true, y_pred, gamma=2.0, alpha=0.25):
    epsilon = K.epsilon()
    y_pred = K.clip(y_pred, epsilon, 1. - epsilon)

    ce = -y_true * K.log(y_pred)
    p_t = K.sum(y_true * y_pred, axis=-1, keepdims=True)
    focal_weight = K.pow((1 - p_t), gamma)
    alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
    focal_loss = alpha_t * focal_weight * ce

    return K.mean(K.sum(focal_loss, axis=-1))

def load_lstm_model(model_folder, model_class, custom_objects=None):
    try:
        print(f"📂 Loading model from: {model_folder}")

        # Load keras model
        keras_model = load_model(f"{model_folder}/keras_model.keras", custom_objects=custom_objects)

        # Load config
        with open(f"{model_folder}/model_config.json", 'r') as f:
            config = json.load(f)

        # Load full model dict
        with open(f"{model_folder}/full_model.pkl", 'rb') as f:
            model_dict = pickle.load(f)

        # Create instance
        model = model_class(pd.DataFrame(), config['sequence_length'], config['forecast_horizon'])
        model.model = keras_model

        # ✅ Conditionally load scalers only if they exist
        scaler_path = os.path.join(model_folder, 'scaler_features.pkl')
        if os.path.exists(scaler_path):
            model.scaler_features = joblib.load(scaler_path)
        else:
            model.scaler_features = None

        scaler_target_path = os.path.join(model_folder, 'scaler_targets.pkl')
        if os.path.exists(scaler_target_path):
            model.scaler_targets = joblib.load(scaler_target_path)
        else:
            model.scaler_targets = None

        for key, value in model_dict.items():
            setattr(model, key, value)

        print(f"✅ Model '{config['model_name']}' successfully loaded!")
        return model

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None
def load_classification_model(model_folder, model_class):
    try:
        print(f"📂 Loading classification model from: {model_folder}")

        # Load Keras model
        keras_model = load_model(
            f"{model_folder}/keras_model.keras",
            custom_objects={'focal_loss_fn': focal_loss_fn}
        )

        # Load label encoder
        with open(f"{model_folder}/label_encoder.pkl", 'rb') as f:
            label_encoder = pickle.load(f)

        # Load scaler
        with open(f"{model_folder}/scaler.pkl", 'rb') as f:
            scaler = pickle.load(f)

        # Load config
        with open(f"{model_folder}/model_config.json", 'r') as f:
            config = json.load(f)

        # Load essential attributes
        with open(f"{model_folder}/essential_attributes.pkl", 'rb') as f:
            essential_attributes = pickle.load(f)

        # Reconstruct model instance
        model = model_class(df=pd.DataFrame(), sequence_length=config['sequence_length'], forecast_horizon=config['forecast_horizon'])
        model.model = keras_model
        model.label_encoder = label_encoder
        model.scaler_X = scaler

        # Restore attributes
        for attr, value in essential_attributes.items():
            setattr(model, attr, value)

        print(f"✅ Classification model '{config['model_name']}' successfully loaded!")
        return model

    except Exception as e:
        print(f"❌ Error loading classification model: {e}")
        return None

In [9]:
class AgriculturalRecommendationSystem:
    def __init__(self):
        """
        Initialize the Agricultural Recommendation System
        """
        self.weather_conditions = {
            'Clear Sky': '☀️',
            'Cloudy': '☁️', 
            'Heavy Drizzle': '🌧',
            'Heavy Rain': '🌧',
            'Light Drizzle': '🌦',
            'Light Rain': '🌦',
            'Moderate Drizzle': '🌧',
            'Moderate Rain': '🌧',
            'Partly Clear': '🌤'
        }
        
        # Agricultural knowledge base
        self.crop_requirements = {
            'tomatoes': {'temp_min': 15, 'temp_max': 30, 'humidity_min': 40, 'humidity_max': 70},
            'lettuce': {'temp_min': 10, 'temp_max': 25, 'humidity_min': 50, 'humidity_max': 80},
            'corn': {'temp_min': 18, 'temp_max': 35, 'humidity_min': 45, 'humidity_max': 75},
            'wheat': {'temp_min': 12, 'temp_max': 28, 'humidity_min': 40, 'humidity_max': 65},
            'rice': {'temp_min': 20, 'temp_max': 35, 'humidity_min': 70, 'humidity_max': 90},
            'beans': {'temp_min': 16, 'temp_max': 30, 'humidity_min': 50, 'humidity_max': 75},
            'carrots': {'temp_min': 8, 'temp_max': 24, 'humidity_min': 45, 'humidity_max': 70},
            'potatoes': {'temp_min': 12, 'temp_max': 26, 'humidity_min': 60, 'humidity_max': 80}
        }
        
        # Add attribute to store the last date from input data
        self.last_input_date = None
        
    def set_last_input_date(self, last_date):
        """
        Set the last date from the input data to calculate correct forecast dates
        
        Args:
            last_date: The last date from your input data (string or datetime object)
        """
        if isinstance(last_date, str):
            # Try to parse the date string
            try:
                # Try common date formats
                for fmt in ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%d %H:%M:%S']:
                    try:
                        self.last_input_date = datetime.strptime(last_date, fmt)
                        break
                    except ValueError:
                        continue
                else:
                    raise ValueError(f"Could not parse date: {last_date}")
            except Exception as e:
                print(f"⚠️ Error parsing date '{last_date}': {e}")
                self.last_input_date = datetime.now()
        elif isinstance(last_date, datetime):
            self.last_input_date = last_date
        else:
            print(f"⚠️ Invalid date type: {type(last_date)}, using current date")
            self.last_input_date = datetime.now()
            
        print(f"📅 Last input date set to: {self.last_input_date.strftime('%Y-%m-%d')}")

    def load_models(self, model_paths: Dict[str, str], processed_data: Dict[str, dict]):
        """Load all models with proper feature column mapping"""
        self.models = {}

        model_class_map = {
            'temp_min': OptimizedTemperatureMinLSTM,
            'temp_max': OptimizedTemperatureMaxLSTM,
            'humidity_min': OptimizedHumidityMinLSTM,
            'humidity_max': OptimizedHumidityMaxLSTM,
            'weather_condition': ImprovedSlidingWindowModel
        }

        for model_name, path in model_paths.items():
            print(f"🔄 Loading model for: {model_name} from {path}")
            model_class = model_class_map.get(model_name)
            if model_class is None:
                print(f"⚠️ Warning: No model class mapped for {model_name}, skipping.")
                continue

            if model_name == 'weather_condition':
                model = load_classification_model(path, model_class)
            else:
                model = load_lstm_model(path, model_class)

            if model is not None:
                # Assign feature columns and scaler if available
                if model_name in processed_data:
                    model.feature_cols = processed_data[model_name]['feature_cols']
                    print(f"  📊 Feature columns for {model_name}: {len(model.feature_cols)} features")
                    
                    # Some models have scaler saved, if available in processed_data:
                    if 'scaler' in processed_data[model_name]:
                        model.scaler_X = processed_data[model_name]['scaler']
                        print(f"  📏 Scaler loaded for {model_name}")
                    else:
                        # For regression models, scaler might not be saved, handle accordingly
                        model.scaler_X = None
                        print(f"  ⚠️ No scaler found for {model_name}")
                else:
                    print(f"⚠️ Warning: No processed data found for {model_name} to set feature_cols/scaler.")

                self.models[model_name] = model
                print(f"  ✅ Model {model_name} loaded successfully")
            else:
                print(f"❌ Failed to load model for {model_name}")

        print("\n✅ All models loaded successfully!")

    def predict_weather(self, input_data, days_ahead: int = 3):
        """Predict weather with proper feature filtering for each model"""
        predictions = {}
        
        print(f"🔮 Starting weather prediction for {days_ahead} days...")
        print(f"📊 Input data shape: {input_data.shape}")
        print(f"📋 Available input features: {len(self.input_feature_names)}")

        for key, model in self.models.items():
            print(f"\n🔄 Processing model: {key}")
            
            if model is None:
                raise ValueError(f"Model for {key} not loaded.")

            # Get model's expected features
            expected_features = model.feature_cols
            print(f"  📊 Model expects {len(expected_features)} features")
            
            # Find indices of expected features in input data
            try:
                model_feature_indices = []
                missing_features = []
                
                for feature in expected_features:
                    if feature in self.input_feature_names:
                        idx = self.input_feature_names.index(feature)
                        model_feature_indices.append(idx)
                    else:
                        missing_features.append(feature)
                
                if missing_features:
                    print(f"  ⚠️ Missing features: {missing_features}")
                    print(f"  📋 Available features: {self.input_feature_names}")
                    raise ValueError(f"Missing required features for {key}: {missing_features}")
                
                print(f"  ✅ Found all {len(model_feature_indices)} required features")
                
                # Filter input data to match model's expected features
                filtered_input = input_data[:, :, model_feature_indices]
                print(f"  📊 Filtered input shape: {filtered_input.shape}")
                
                # Convert to DataFrame for LSTM models (they expect DataFrame input)
                if key != 'weather_condition':
                    # Create DataFrame with proper column names
                    last_sequence = filtered_input[0]  # Shape: (seq_len, features)
                    df_input = pd.DataFrame(last_sequence, columns=expected_features)
                    print(f"  📊 Created DataFrame input shape: {df_input.shape}")
                    
                    # Make prediction using DataFrame
                    pred = model.predict_future(df_input)
                    
                    # Ensure pred is the right shape
                    if isinstance(pred, np.ndarray):
                        if pred.ndim == 1:
                            pred = pred[:days_ahead]
                        elif pred.ndim == 2:
                            pred = pred[0, :days_ahead]
                    
                else:
                    # For classification model, handle differently
                    # Extract the last sequence for prediction
                    last_sequence = filtered_input[0]  # Shape: (seq_len, features)
                    pred = model.predict_future(last_sequence)
                    
                    # pred might be a tuple (conditions, confidence, entropy)
                    if isinstance(pred, tuple):
                        pred = pred[0]  # Take the predicted conditions
                    
                    # Ensure we have the right number of predictions
                    if len(pred) > days_ahead:
                        pred = pred[:days_ahead]
                    elif len(pred) < days_ahead:
                        # Repeat last prediction if needed
                        pred = np.concatenate([pred, [pred[-1]] * (days_ahead - len(pred))])
                
                predictions[key] = pred
                print(f"  ✅ Prediction completed for {key}")
                print(f"  📊 Prediction shape: {np.array(pred).shape}")
                    
            except Exception as e:
                print(f"  ❌ Error processing {key}: {str(e)}")
                import traceback
                traceback.print_exc()
                raise

        print(f"\n✅ All predictions completed!")
        return predictions

    def set_input_feature_names(self, feature_names: List[str]):
        """Set the list of input feature names used for filtering per-model inputs."""
        self.input_feature_names = feature_names
        print(f"📋 Set {len(feature_names)} input feature names")

    def analyze_irrigation_needs(self, weather_condition: str, humidity_min: float, 
                               humidity_max: float, temp_max: float) -> Dict[str, Any]:
        irrigation_advice = {
            'priority': 'medium',
            'frequency': 'normal',
            'amount': 'moderate',
            'timing': 'morning',
            'notes': []
        }
        
        # Analyze based on weather condition
        if weather_condition in ['Heavy Rain', 'Moderate Rain', 'Heavy Drizzle', 'Moderate Drizzle']:
            irrigation_advice['priority'] = 'low'
            irrigation_advice['frequency'] = 'reduced'
            irrigation_advice['amount'] = 'minimal'
            irrigation_advice['notes'].append('Natural rainfall expected - reduce irrigation')
            
        elif weather_condition in ['Clear Sky', 'Partly Clear']:
            if temp_max > 30:
                irrigation_advice['priority'] = 'high'
                irrigation_advice['frequency'] = 'increased'
                irrigation_advice['amount'] = 'generous'
                irrigation_advice['notes'].append('Hot and sunny - increase watering')
            
        elif weather_condition == 'Cloudy':
            irrigation_advice['priority'] = 'medium'
            irrigation_advice['notes'].append('Moderate conditions - maintain regular schedule')
        
        # Adjust based on humidity
        if humidity_min < 40:
            irrigation_advice['priority'] = 'high'
            irrigation_advice['notes'].append('Low humidity - plants need extra water')
        elif humidity_max > 80:
            irrigation_advice['frequency'] = 'reduced'
            irrigation_advice['notes'].append('High humidity - reduce watering frequency')
        
        return irrigation_advice
    
    def analyze_pest_disease_risk(self, weather_condition: str, humidity_max: float, 
                                temp_max: float, temp_min: float) -> Dict[str, Any]:
        risk_assessment = {
            'overall_risk': 'medium',
            'pest_risk': 'medium',
            'disease_risk': 'medium',
            'specific_threats': [],
            'preventive_measures': []
        }
        
        # High humidity and warm temperatures increase disease risk
        if humidity_max > 75 and temp_max > 25:
            risk_assessment['disease_risk'] = 'high'
            risk_assessment['overall_risk'] = 'high'
            risk_assessment['specific_threats'].extend(['Fungal diseases', 'Bacterial infections'])
            risk_assessment['preventive_measures'].extend([
                'Improve air circulation',
                'Apply preventive fungicides',
                'Monitor for early symptoms'
            ])
        
        # Rain conditions can increase disease pressure
        if weather_condition in ['Heavy Rain', 'Moderate Rain', 'Heavy Drizzle']:
            risk_assessment['disease_risk'] = 'high'
            risk_assessment['specific_threats'].append('Soil-borne diseases')
            risk_assessment['preventive_measures'].append('Ensure proper drainage')
        
        # Hot, dry conditions may increase pest activity
        if weather_condition == 'Clear Sky' and temp_max > 30 and humidity_max < 50:
            risk_assessment['pest_risk'] = 'high'
            risk_assessment['specific_threats'].extend(['Aphids', 'Spider mites', 'Thrips'])
            risk_assessment['preventive_measures'].extend([
                'Monitor for pest presence',
                'Maintain adequate moisture',
                'Consider beneficial insects'
            ])
        
        return risk_assessment
    
    def recommend_planting_activities(self, weather_condition: str, temp_max: float, 
                                    temp_min: float, humidity_avg: float) -> Dict[str, Any]:
        recommendations = {
            'planting_suitability': 'moderate',
            'recommended_crops': [],
            'field_activities': [],
            'avoid_activities': [],
            'timing_advice': []
        }
        
        # Analyze temperature conditions
        if temp_min < 10:
            recommendations['planting_suitability'] = 'poor'
            recommendations['avoid_activities'].append('Planting heat-sensitive crops')
            recommendations['timing_advice'].append('Wait for warmer conditions')
        elif temp_min > 15 and temp_max < 30:
            recommendations['planting_suitability'] = 'excellent'
            recommendations['recommended_crops'].extend(['tomatoes', 'beans', 'corn'])
        
        # Weather-specific recommendations
        if weather_condition in ['Heavy Rain', 'Moderate Rain']:
            recommendations['avoid_activities'].extend([
                'Planting seeds (risk of rot)',
                'Harvesting',
                'Soil cultivation'
            ])
            recommendations['timing_advice'].append('Wait for soil to dry before field work')
            
        elif weather_condition == 'Clear Sky':
            recommendations['field_activities'].extend([
                'Planting',
                'Transplanting',
                'Harvesting',
                'Soil preparation'
            ])
            recommendations['timing_advice'].append('Ideal conditions for most field activities')
        
        # Recommend suitable crops based on conditions
        for crop, requirements in self.crop_requirements.items():
            if (requirements['temp_min'] <= temp_min <= requirements['temp_max'] and
                requirements['humidity_min'] <= humidity_avg <= requirements['humidity_max']):
                if crop not in recommendations['recommended_crops']:
                    recommendations['recommended_crops'].append(crop)
        
        return recommendations
    
    def generate_comprehensive_report(self, predictions: Dict[str, Any], days_ahead: int = 7) -> str:
        report = []
        report.append("🌱 AGRICULTURAL WEATHER RECOMMENDATION REPORT 🌱")
        report.append("=" * 50)
        
        # Use the last input date if available, otherwise use current date
        if self.last_input_date:
            base_date = self.last_input_date
            report.append(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            report.append(f"Based on data up to: {base_date.strftime('%Y-%m-%d')}")
        else:
            base_date = datetime.now()
            report.append(f"Generated on: {base_date.strftime('%Y-%m-%d %H:%M:%S')}")
            print("⚠️ Warning: No last input date set, using current date for forecasts")
        
        report.append(f"Forecast period: {days_ahead} days\n")
        
        # Daily analysis
        for day in range(days_ahead):
            day_date = base_date + timedelta(days=day+1)
            weather_cond = predictions['weather_condition'][day]
            temp_max = predictions['temp_max'][day]
            temp_min = predictions['temp_min'][day]
            humidity_max = predictions['humidity_max'][day]
            humidity_min = predictions['humidity_min'][day]
            humidity_avg = (humidity_max + humidity_min) / 2
            
            report.append(f"📅 DAY {day+1} - {day_date.strftime('%A, %B %d, %Y')}")
            report.append("-" * 30)
            report.append(f"Weather: {weather_cond} {self.weather_conditions.get(weather_cond, '🌤')}")
            report.append(f"Temperature: {temp_min:.1f}°C - {temp_max:.1f}°C")
            report.append(f"Humidity: {humidity_min:.1f}% - {humidity_max:.1f}%\n")
            
            # Irrigation analysis
            irrigation = self.analyze_irrigation_needs(weather_cond, humidity_min, humidity_max, temp_max)
            report.append(f"💧 IRRIGATION ADVICE:")
            report.append(f"  Priority: {irrigation['priority'].upper()}")
            report.append(f"  Frequency: {irrigation['frequency']}")
            report.append(f"  Amount: {irrigation['amount']}")
            report.append(f"  Best timing: {irrigation['timing']}")
            for note in irrigation['notes']:
                report.append(f"  • {note}")
            report.append("")
            
            # Pest and disease risk
            risk = self.analyze_pest_disease_risk(weather_cond, humidity_max, temp_max, temp_min)
            report.append(f"🐛 PEST & DISEASE RISK:")
            report.append(f"  Overall risk: {risk['overall_risk'].upper()}")
            report.append(f"  Pest risk: {risk['pest_risk']}")
            report.append(f"  Disease risk: {risk['disease_risk']}")
            if risk['specific_threats']:
                report.append(f"  Threats: {', '.join(risk['specific_threats'])}")
            if risk['preventive_measures']:
                report.append(f"  Prevention:")
                for measure in risk['preventive_measures']:
                    report.append(f"    • {measure}")
            report.append("")
            
            # Planting recommendations
            planting = self.recommend_planting_activities(weather_cond, temp_max, temp_min, humidity_avg)
            report.append(f"🌱 PLANTING & FIELD ACTIVITIES:")
            report.append(f"  Planting conditions: {planting['planting_suitability'].upper()}")
            if planting['recommended_crops']:
                report.append(f"  Suitable crops: {', '.join(planting['recommended_crops'])}")
            if planting['field_activities']:
                report.append(f"  Recommended activities: {', '.join(planting['field_activities'])}")
            if planting['avoid_activities']:
                report.append(f"  Avoid: {', '.join(planting['avoid_activities'])}")
            for advice in planting['timing_advice']:
                report.append(f"  • {advice}")
            
            report.append("\n" + "="*50 + "\n")
        
        # Weekly summary
        report.append("📊 WEEKLY SUMMARY & RECOMMENDATIONS")
        report.append("=" * 50)
        
        # Calculate weekly averages
        avg_temp_max = np.mean(predictions['temp_max'])
        avg_temp_min = np.mean(predictions['temp_min'])
        avg_humidity = np.mean([predictions['humidity_max'], predictions['humidity_min']])
        
        rainy_days = sum(1 for cond in predictions['weather_condition'] 
                        if 'Rain' in cond or 'Drizzle' in cond)
        
        report.append(f"Average temperatures: {avg_temp_min:.1f}°C - {avg_temp_max:.1f}°C")
        report.append(f"Average humidity: {avg_humidity:.1f}%")
        report.append(f"Rainy days expected: {rainy_days}/{days_ahead}")
        report.append("")
        
        # Weekly recommendations
        report.append("🎯 KEY WEEKLY RECOMMENDATIONS:")
        if rainy_days >= 3:
            report.append("• High rainfall expected - focus on drainage and disease prevention")
        if avg_temp_max > 30:
            report.append("• Hot week ahead - increase irrigation and provide shade where possible")
        if avg_humidity > 75:
            report.append("• High humidity - monitor closely for fungal diseases")
        if rainy_days <= 1 and avg_temp_max > 25:
            report.append("• Dry conditions - excellent for harvesting and field work")
        
        report.append("\n" + "="*50)
        report.append("Generated by Agricultural Weather Recommendation System")
        
        return "\n".join(report)
    
    def save_report(self, recommendations, filename="agricultural_recommendations.txt", 
                   encoding='utf-8', include_emojis=True):
        """
        Save the recommendations report with proper encoding handling
        
        Args:
            recommendations: The report text to save
            filename: Output filename
            encoding: Text encoding ('utf-8', 'ascii', etc.)
            include_emojis: Whether to include emoji characters
        """
        try:
            text_to_save = recommendations
            
            if not include_emojis:
                text_to_save = remove_emojis(text_to_save)
            
            with open(filename, 'w', encoding=encoding, errors='ignore') as f:
                f.write(text_to_save)
            
            print(f"\n📄 Report saved to '{filename}' with {encoding} encoding")
            return True
            
        except Exception as e:
            print(f"❌ Error saving file: {str(e)}")
            return False
    
    def get_recommendations_with_safe_save(self, input_data: np.ndarray, 
                                         days_ahead: int = 3, 
                                         save_file: bool = True,
                                         filename: str = "agricultural_recommendations.txt") -> str:
        """Generate recommendations and save with proper encoding"""
        print(f"🌾 Generating agricultural recommendations for {days_ahead} days...")
        
        # Get predictions from your models
        predictions = self.predict_weather(input_data, days_ahead)
        
        # Generate comprehensive report
        report = self.generate_comprehensive_report(predictions, days_ahead)
        
        # Save with proper encoding if requested
        if save_file:
            # Try UTF-8 first, fall back to ASCII if needed
            if not self.save_report(report, filename, encoding='utf-8', include_emojis=True):
                print("🔄 Retrying with ASCII encoding...")
                self.save_report(report, filename, encoding='ascii', include_emojis=False)
        
        return report

In [11]:
# Updated main execution code with proper date handling
if __name__ == "__main__":
    import pandas as pd
    import numpy as np
    from pathlib import Path
    from datetime import datetime

    # Load raw data
    df = pd.read_csv("test_data.csv")
    
    # Get the last date from your data
    # Assuming your CSV has a 'date' column
    if 'date' in df.columns:
        last_date = df['date'].iloc[-1]  # Get the last date
        print(f"📅 Last date in data: {last_date}")
    else:
        # If no date column, you need to specify it manually
        last_date = "2025-05-15"  # Your last date
        print(f"📅 Manually set last date: {last_date}")

    # Initialize and transform with feature engineer
    engineer = WeatherFeatureEngineer()
    df_transformed = engineer.transform(df)

    # Prepare models
    models = ['temp_max', 'temp_min', 'humidity_max', 'humidity_min', 'weather_condition']
    processed_data = {}

    for model_type in models:
        print(f"\nProcessing data for model: {model_type}")
        if model_type == 'weather_condition':
            X, y, feature_cols, label_encoder, class_weight_dict, scaler = engineer.generate_model_inputs(df_transformed, model_type)
            processed_data[model_type] = {
                'X': X,
                'y': y,
                'feature_cols': feature_cols,
                'label_encoder': label_encoder,
                'class_weight_dict': class_weight_dict,
                'scaler': scaler,
            }
            print(f"  📊 Features for {model_type}: {len(feature_cols)}")
        else:
            X_scaled, feature_cols = engineer.generate_model_inputs(df_transformed, model_type)
            processed_data[model_type] = {
                'X': X_scaled,
                'feature_cols': feature_cols,
            }
            print(f"  📊 Features for {model_type}: {len(feature_cols)}")

    print("\n✅ All model inputs prepared and ready.")

    # ============================
    # 🎯 Use in Agricultural System
    # ============================
    agri_system = AgriculturalRecommendationSystem()
    
    # Set the last input date BEFORE making predictions
    agri_system.set_last_input_date(last_date)

    model_paths = {
        'weather_condition': 'saved_models/BestWeatherModel_20250704_070814',
        'humidity_min': 'saved_models/BestMinimumHumidityLSTM_Model1_20250704_150613',
        'humidity_max': 'saved_models/BestMaximumHumidityLSTM_Model1_20250704_134945',
        'temp_min': 'saved_models/BestMinimumTemperatureLSTM_Model1_20250704_122946',
        'temp_max': 'saved_models/BestMaximumTemperatureLSTM_Model1_20250704_110121',
    }

    # Load models with processed data
    agri_system.load_models(model_paths, processed_data)

    # Extract input feature names from your feature engineer's transformed df
    exclude_cols = ['date', 'weather_condition']
    input_feature_names = [col for col in df_transformed.columns if col not in exclude_cols]
    
    print(f"\n📋 Available input features ({len(input_feature_names)}): {input_feature_names[:10]}...")  # Show first 10
    
    agri_system.set_input_feature_names(input_feature_names)

    # Get sequence_length from one model
    sample_model = list(agri_system.models.values())[0]
    seq_len = sample_model.sequence_length
    print(f"📏 Sequence length: {seq_len}")

    # Prepare input data correctly
    print(f"\n🔧 Preparing input data...")
    
    # Get the last sequence from the transformed data
    feature_data = df_transformed[input_feature_names]
    
    # Take the last seq_len rows
    input_sequence = feature_data.iloc[-seq_len:].values
    print(f"📊 Input sequence shape: {input_sequence.shape}")
    
    # Add batch dimension
    input_data = np.expand_dims(input_sequence, axis=0)
    print(f"📊 Final input shape: {input_data.shape}")

    # Get recommendations
    try:
        recommendations = agri_system.get_recommendations_with_safe_save(
            input_data, days_ahead=3, save_file=True
        )
        print(recommendations)
        
    except Exception as e:
        print(f"\n❌ Error generating recommendations: {str(e)}")
        import traceback
        traceback.print_exc()

📅 Last date in data: 2025-05-15
Starting feature transformation pipeline...
DEBUG: is_weekend created. Shape: (15,), dtype: int64
DEBUG: is_weekend values: [0 1]
Dropping 7 unimportant columns: ['dew_point_2m_max (°C)', 'wind_speed_10m_max (km/h)', 'surface_pressure_max (hPa)', 'surface_pressure_min (hPa)', 'wet_bulb_temperature_2m_max (°C)', 'wet_bulb_temperature_2m_min (°C)', 'soil_temperature_0_to_100cm_mean (°C)']
Creating targeted weather features based on importance analysis...
1. Engineering Tier 1 (Core Weather State) features...
2. Engineering Tier 2 (Physical Variables) features...
3. Creating interaction features...
4. Creating weather pattern features...
5. Optimizing feature set...
Removing 1 low-importance non-temporal features...

TARGETED FEATURE ENGINEERING SUMMARY
Original features: 40
Final features: 150
New features added: 110
Features removed: 1

Feature Categories:
  Weather State (Tier 1): ~81 features
  Physical Variables (Tier 2): ~30 features
  Temporal Featur